In [1]:
# ============================================================
# CELL 1 — IMPORT LIBRARIES
# ============================================================

import os
import datetime
import joblib

import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score, )

print('cell 1, ok')

cell 1, ok


In [2]:
# ============================================================
# CELL 2 — READ AND PREPROCESS MODEL HGBR INPUT DATA
# ============================================================

import os
import numpy as np
from datetime import datetime

sourcepath = (r"C:\ML\Data_Learn")
sourcename_tec = ("dataset_learning_VTEC-15-min_ptn.txt")
sourcename_dst = ("dataset_learning_dst-index.txt")
sourcename_f107 = ("dataset_learning_F10-7.txt")
sourcename_bz = ("dataset_learning_IMFBz-15-min.txt")
sourcename_vsw = ("dataset_learning_Vsw-15-min.txt")

def merge_duplicate_time(data_time, data_value):
    unique_time, inverse = np.unique(data_time, return_inverse=True)
    clean_value = np.full(len(unique_time), np.nan, dtype=np.float32)
    for i in range(len(unique_time)):
        values = data_value[inverse == i]
        if np.any(np.isfinite(values)):
            clean_value[i] = np.nanmean(values)
    return (unique_time, clean_value)


def print_data_information(title, filename, data_time, data_value):
    print(title)
    print(f"File       : {filename}")
    print(f"Total data : {len(data_time)}")
    if len(data_time) > 0:        
        print(f"Start time : {data_time[0]}")
        print(f"End time   : {data_time[-1]}")
    print(f"NaN        : " f"{np.sum(~np.isfinite(data_value))}")

file_tec = os.path.join(sourcepath, sourcename_tec)
if not os.path.isfile(file_tec):    
    raise FileNotFoundError("VTEC file not found:\n" + file_tec)

data_time_tec = []
data_value_tec = []

with open(file_tec, "r", encoding="utf-8") as f:
    for line in f:        
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) < 7:
            continue
        try:
            year = int(parts[0])
            month = int(parts[1])
            day = int(parts[2])
            hour = int(parts[3])
            minute = int(parts[4])
            second = int(parts[5])

            timestamp = datetime(year, month, day, hour, minute, second)
            try:
                value = float(parts[6])
            except ValueError:
                value = np.nan

            data_time_tec.append(timestamp)
            data_value_tec.append(value)

        except ValueError:
            continue

data_time_tec = np.array(data_time_tec, dtype="datetime64[s]")
data_value_tec = np.array(data_value_tec, dtype=np.float32)

sort_idx = np.argsort(data_time_tec)
data_time_tec = (data_time_tec[sort_idx])
data_value_tec = (data_value_tec[sort_idx])

# VTEC INFORMATION BEFORE DUPLICATE
print_data_information(title="VTEC DATA - BEFORE DUPLICATE HANDLING", filename=sourcename_tec,
                       data_time=data_time_tec, data_value=data_value_tec, )

# VTEC DUPLICATE CHECK
unique_time_check, counts = np.unique(data_time_tec, return_counts=True)
duplicate_count_tec = int(np.sum(counts > 1))

print(f"Duplicate timestamp : " f"{duplicate_count_tec}")

# MERGE VTEC DUPLICATE
(data_time_tec, data_value_tec) = merge_duplicate_time(data_time_tec, data_value_tec)

# CREATE CONTINUOUS VTEC 15-MINUTE GRID
if len(data_time_tec) == 0:
    raise ValueError("VTEC dataset is empty.")

start_time_tec = (data_time_tec[0])
end_time_tec = (data_time_tec[-1])
time_grid_tec = np.arange(start_time_tec, end_time_tec + np.timedelta64(15, "m"), np.timedelta64(15, "m"))


# MAP VTEC DATA TO GRID
value_grid_tec = np.full(len(time_grid_tec), np.nan, dtype=np.float32)
time_index_tec = {timestamp: i for i, timestamp in enumerate(data_time_tec)}

for i, timestamp in enumerate(time_grid_tec):
    if timestamp in time_index_tec:
        original_idx = (time_index_tec[timestamp])

        value_grid_tec[i] = (data_value_tec[original_idx])


# FINAL VTEC
data_time_tec = (time_grid_tec)
data_value_tec = (value_grid_tec)

# VTEC INTERVAL CHECK
if len(data_time_tec) > 1:
    interval_tec = np.diff(data_time_tec)
    unique_interval_tec = (np.unique(interval_tec))

    print("VTEC unique time interval:")
    for interval in (unique_interval_tec):
        print(interval)    
        
        expected_interval_tec = (np.timedelta64(15, "m"))

    if np.all(interval_tec == expected_interval_tec):
        print("VTEC time interval check : " "OK (15 minutes)")

    else:
        print("VTEC time interval check : " "WARNING")


# VTEC FINAL INFORMATION
print()
print("VTEC DATA - FINAL")

print(f"Total timestamp : " f"{len(data_time_tec)}")
print(f"Start time      : " f"{data_time_tec[0]}")
print(f"End time        : " f"{data_time_tec[-1]}")
print(f"NaN             : " f"{np.sum(~np.isfinite(data_value_tec))}")
print(f"Valid           : " f"{np.sum(np.isfinite(data_value_tec))}")


# READ DST
file_dst = os.path.join(sourcepath, sourcename_dst)
if not os.path.isfile(file_dst):
    raise FileNotFoundError("Dst file not found:\n" + file_dst)

data_time_dst = []
data_value_dst = []
with open(file_dst, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue

        parts = line.split()
        if len(parts) < 7:
            continue

        try:
            year = int(parts[0])
            month = int(parts[1])
            day = int(parts[2])
            hour = int(parts[3])
            minute = int(parts[4])
            second = int(parts[5])
            timestamp = datetime(year, month, day, hour, minute, second)

            try:
                value = float(parts[6])

            except ValueError:
                value = np.nan
            data_time_dst.append(timestamp)
            data_value_dst.append(value)

        except ValueError:
            continue


# CONVERT TO NUMPY
data_time_dst = np.array(data_time_dst, dtype="datetime64[s]")
data_value_dst = np.array(data_value_dst, dtype=np.float32)


# SORT DST
sort_idx_dst = np.argsort(data_time_dst)
data_time_dst = (data_time_dst[sort_idx_dst])
data_value_dst = (data_value_dst[sort_idx_dst])

# DST INFORMATION BEFORE DUPLICATE
print_data_information(title="DST DATA - BEFORE DUPLICATE HANDLING", 
                       filename=sourcename_dst, data_time=data_time_dst, data_value=data_value_dst,)

# DST DUPLICATE CHECK
unique_time_dst, counts_dst = np.unique(data_time_dst, return_counts=True)
duplicate_count_dst = int(np.sum(counts_dst > 1))
print(f"Duplicate timestamp : " f"{duplicate_count_dst}")

# MERGE DST DUPLICATE
(data_time_dst, data_value_dst) = merge_duplicate_time(data_time_dst, data_value_dst)


# CREATE CONTINUOUS DST 1-HOUR GRID
if len(data_time_dst) == 0:
    raise ValueError("Dst dataset is empty.")

start_time_dst = (data_time_dst[0])
end_time_dst = (data_time_dst[-1])
time_grid_dst = np.arange(start_time_dst, end_time_dst + np.timedelta64(1, "h"), np.timedelta64(1, "h"))

# MAP DST DATA TO GRID
value_grid_dst = np.full(len(time_grid_dst), np.nan, dtype=np.float32)
time_index_dst = {timestamp: i for i, timestamp in enumerate(data_time_dst)}

for i, timestamp in enumerate(time_grid_dst):
    if timestamp in time_index_dst:
        original_idx = (time_index_dst[timestamp])

        value_grid_dst[i] = (data_value_dst[original_idx])

# FINAL DST
data_time_dst = (time_grid_dst)
data_value_dst = (value_grid_dst)

# DST INTERVAL CHECK
if len(data_time_dst) > 1:
    interval_dst = np.diff(data_time_dst)
    unique_interval_dst = (np.unique(interval_dst))

    print()
    print("Dst unique time interval:")

    for interval in (unique_interval_dst):
        print(interval)
    expected_interval_dst = (np.timedelta64(1, "h"))

    if np.all(interval_dst == expected_interval_dst):
        print("Dst time interval check : " "OK (1 hour)")

    else:
        print("Dst time interval check : " "WARNING")


# DST FINAL INFORMATION
print()
print("DST DATA - FINAL")
print(f"Total timestamp : " f"{len(data_time_dst)}")
print(f"Start time      : " f"{data_time_dst[0]}")
print(f"End time        : " f"{data_time_dst[-1]}")
print(f"NaN             : " f"{np.sum(~np.isfinite(data_value_dst))}")
print(f"Valid           : " f"{np.sum(np.isfinite(data_value_dst))}")

# READ F10.7
file_f107 = os.path.join(sourcepath, sourcename_f107)
if not os.path.isfile(file_f107):
    raise FileNotFoundError("F10.7 file not found:\n" + file_f107)

data_time_f107 = []
data_value_f107 = []
with open(file_f107, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()

        if not line:
            continue

        parts = line.split()
        if len(parts) < 4:
            continue

        try:
            year = int(parts[0])
            month = int(parts[1])
            day = int(parts[2])
            timestamp = datetime( year, month, day)

            try:
                value = float(parts[3])

            except ValueError:
                value = np.nan

            data_time_f107.append(timestamp)
            data_value_f107.append(value)

        except ValueError:
            continue

# CONVERT TO NUMPY
data_time_f107 = np.array(data_time_f107, dtype="datetime64[D]")
data_value_f107 = np.array(data_value_f107, dtype=np.float32)

# SORT F10.7
sort_idx_f107 = np.argsort(data_time_f107)
data_time_f107 = (data_time_f107[sort_idx_f107])
data_value_f107 = (data_value_f107[sort_idx_f107])

# F10.7 INFORMATION BEFORE DUPLICATE
print_data_information(title="F10.7 DATA - BEFORE DUPLICATE HANDLING", 
                       filename=sourcename_f107, data_time=data_time_f107, data_value=data_value_f107, )

# F10.7 DUPLICATE CHECK
unique_time_f107, counts_f107 = np.unique(data_time_f107, return_counts=True)
duplicate_count_f107 = int(np.sum(counts_f107 > 1))
print(f"Duplicate timestamp : " f"{duplicate_count_f107}")

# MERGE F10.7 DUPLICATE
(data_time_f107, data_value_f107) = merge_duplicate_time(data_time_f107, data_value_f107)

# CREATE CONTINUOUS F10.7 1-DAY GRID
if len(data_time_f107) == 0:
    raise ValueError("F10.7 dataset is empty.")

start_time_f107 = (data_time_f107[0])
end_time_f107 = (data_time_f107[-1])
time_grid_f107 = np.arange(start_time_f107, end_time_f107 + np.timedelta64(1, "D"), np.timedelta64(1, "D"))

# MAP F10.7 DATA TO GRID
value_grid_f107 = np.full(len(time_grid_f107), np.nan, dtype=np.float32)
time_index_f107 = {timestamp: i for i, timestamp in enumerate(data_time_f107)}
for i, timestamp in enumerate(time_grid_f107):
    if timestamp in time_index_f107:
        original_idx = (time_index_f107[timestamp])
        value_grid_f107[i] = (data_value_f107[original_idx])

# FINAL F10.7
data_time_f107 = (time_grid_f107)
data_value_f107 = (value_grid_f107)

# F10.7 INTERVAL CHECK
if len(data_time_f107) > 1:
    interval_f107 = np.diff(data_time_f107)
    unique_interval_f107 = (np.unique(interval_f107))

    print()
    print("F10.7 unique time interval:")
    for interval in (unique_interval_f107):
        print(interval)

    expected_interval_f107 = (np.timedelta64(1, "D"))
    if np.all(interval_f107 == expected_interval_f107):
        print("F10.7 time interval check : " "OK (1 day)")

    else:
        print("F10.7 time interval check : " "WARNING")

# F10.7 FINAL INFORMATION
print()
print("F10.7 DATA - FINAL")
print(f"Total timestamp : " f"{len(data_time_f107)}")
print(f"Start time      : " f"{data_time_f107[0]}")
print(f"End time        : " f"{data_time_f107[-1]}")
print(f"NaN             : " f"{np.sum(~np.isfinite(data_value_f107))}")
print(f"Valid           : " f"{np.sum(np.isfinite(data_value_f107))}")


# READ IMF Bz
file_bz = os.path.join(sourcepath, sourcename_bz)
if not os.path.isfile(file_bz):
    raise FileNotFoundError("IMF Bz file not found:\n" + file_bz)

data_time_bz = []
data_value_bz = []
with open(file_bz, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()

        if not line:
            continue

        parts = line.split()
        if len(parts) < 7:
            continue

        try:
            year = int(parts[0])
            month = int(parts[1])
            day = int(parts[2])
            hour = int(parts[3])
            minute = int(parts[4])
            second = int(parts[5])
            timestamp = datetime(year, month, day, hour, minute, second)

            try:
                value = float(parts[6])

            except ValueError:
                value = np.nan

            data_time_bz.append(timestamp)
            data_value_bz.append(value)

        except ValueError:
            continue

# CONVERT TO NUMPY
data_time_bz = np.array(data_time_bz, dtype="datetime64[s]")
data_value_bz = np.array(data_value_bz, dtype=np.float32)

# SORT IMF Bz
sort_idx_bz = np.argsort(data_time_bz)
data_time_bz = (data_time_bz[sort_idx_bz])
data_value_bz = (data_value_bz[sort_idx_bz])

# IMF Bz INFORMATION BEFORE DUPLICATE
print_data_information(title="IMF Bz DATA - BEFORE DUPLICATE HANDLING",
                       filename=sourcename_bz, data_time=data_time_bz, data_value=data_value_bz, )


# IMF Bz DUPLICATE CHECK
unique_time_bz, counts_bz = np.unique(data_time_bz, return_counts=True)
duplicate_count_bz = int(np.sum(counts_bz > 1))
print(f"Duplicate timestamp : " f"{duplicate_count_bz}")

# MERGE IMF Bz DUPLICATE
(data_time_bz, data_value_bz) = merge_duplicate_time(data_time_bz, data_value_bz)

# CREATE CONTINUOUS IMF Bz 15-MINUTE GRID
if len(data_time_bz) == 0:
    raise ValueError("IMF Bz dataset is empty.")

start_time_bz = (data_time_bz[0])
end_time_bz = (data_time_bz[-1])
time_grid_bz = np.arange(start_time_bz, end_time_bz + np.timedelta64(15, "m"), np.timedelta64(15, "m"))

# MAP IMF Bz DATA TO GRID
value_grid_bz = np.full(len(time_grid_bz), np.nan, dtype=np.float32)
time_index_bz = {timestamp: i for i, timestamp in enumerate(data_time_bz)}

for i, timestamp in enumerate(time_grid_bz):
    if timestamp in time_index_bz:
        original_idx = (time_index_bz[timestamp])
        value_grid_bz[i] = (data_value_bz[original_idx])

# FINAL IMF Bz
data_time_bz = (time_grid_bz)
data_value_bz = (value_grid_bz)


# IMF Bz INTERVAL CHECK
if len(data_time_bz) > 1:

    interval_bz = np.diff(data_time_bz)
    unique_interval_bz = (np.unique(interval_bz))

    print()
    print("IMF Bz unique time interval:")
    for interval in (unique_interval_bz):
        print(interval)

    expected_interval_bz = (np.timedelta64(15, "m"))
    if np.all(interval_bz == expected_interval_bz):
        print("IMF Bz time interval check : " "OK (15 minutes)")

    else:
        print("IMF Bz time interval check : " "WARNING")


# IMF Bz FINAL INFORMATION
print()
print("IMF Bz DATA - FINAL")
print(f"Total timestamp : " f"{len(data_time_bz)}")
print(f"Start time      : " f"{data_time_bz[0]}")
print(f"End time        : " f"{data_time_bz[-1]}")
print(f"NaN             : " f"{np.sum(~np.isfinite(data_value_bz))}")
print(f"Valid           : " f"{np.sum(np.isfinite(data_value_bz))}")


# READ Vsw / Vtotal
file_vsw = os.path.join(sourcepath, sourcename_vsw)
if not os.path.isfile(file_vsw):
    raise FileNotFoundError("Vsw/Vtotal file not found:\n" + file_vsw)

data_time_vsw = []
data_value_vsw = []
with open(file_vsw, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue

        parts = line.split()
        if len(parts) < 7:
            continue

        try:
            year = int(parts[0])
            month = int(parts[1])
            day = int(parts[2])
            hour = int(parts[3])
            minute = int(parts[4])
            second = int(parts[5])

            timestamp = datetime(year, month, day, hour, minute, second)

            try:
                value = float(parts[6])

            except ValueError:
                value = np.nan

            data_time_vsw.append(timestamp)
            data_value_vsw.append(value)

        except ValueError:
            continue

# CONVERT TO NUMPY
data_time_vsw = np.array(data_time_vsw, dtype="datetime64[s]")
data_value_vsw = np.array(data_value_vsw, dtype=np.float32)

# SORT Vsw / Vtotal
sort_idx_vsw = np.argsort(data_time_vsw)
data_time_vsw = (data_time_vsw[sort_idx_vsw])
data_value_vsw = (data_value_vsw[sort_idx_vsw])

# Vsw / Vtotal INFORMATION BEFORE DUPLICATE
print_data_information(title="Vsw / Vtotal DATA - BEFORE DUPLICATE HANDLING",
                       filename=sourcename_vsw, data_time=data_time_vsw, data_value=data_value_vsw, )


# Vsw / Vtotal DUPLICATE CHECK
unique_time_vsw, counts_vsw = np.unique(data_time_vsw, return_counts=True)
duplicate_count_vsw = int(np.sum(counts_vsw > 1))
print(f"Duplicate timestamp : " f"{duplicate_count_vsw}")

# MERGE Vsw / Vtotal DUPLICATE
(data_time_vsw, data_value_vsw) = merge_duplicate_time(data_time_vsw, data_value_vsw)

# CREATE CONTINUOUS Vsw / Vtotal 15-MINUTE GRID
if len(data_time_vsw) == 0:
    raise ValueError("Vsw/Vtotal dataset is empty.")

start_time_vsw = (data_time_vsw[0])
end_time_vsw = (data_time_vsw[-1])
time_grid_vsw = np.arange(start_time_vsw, end_time_vsw + np.timedelta64(15, "m"), np.timedelta64(15, "m"))

# MAP Vsw / Vtotal DATA TO GRID
value_grid_vsw = np.full(len(time_grid_vsw), np.nan, dtype=np.float32)
time_index_vsw = {timestamp: i for i, timestamp in enumerate(data_time_vsw)}

for i, timestamp in enumerate(time_grid_vsw):
    if timestamp in time_index_vsw:
        original_idx = (time_index_vsw[timestamp])

        value_grid_vsw[i] = (data_value_vsw[original_idx])


# FINAL Vsw / Vtotal
data_time_vsw = (time_grid_vsw)
data_value_vsw = (value_grid_vsw)

# Vsw / Vtotal INTERVAL CHECK
if len(data_time_vsw) > 1:
    interval_vsw = np.diff(data_time_vsw)
    unique_interval_vsw = (np.unique(interval_vsw))

    print()
    print("Vsw/Vtotal unique time interval:")

    for interval in (unique_interval_vsw):
        print(interval)

    expected_interval_vsw = (np.timedelta64(15, "m"))
    if np.all(interval_vsw == expected_interval_vsw):
        print("Vsw/Vtotal time interval check : " "OK (15 minutes)")

    else:
        print("Vsw/Vtotal time interval check : " "WARNING")


# Vsw / Vtotal FINAL INFORMATION
print()
print("Vsw / Vtotal DATA - FINAL")
print(f"Total timestamp : " f"{len(data_time_vsw)}")
print(f"Start time      : " f"{data_time_vsw[0]}")
print(f"End time        : " f"{data_time_vsw[-1]}")
print(f"NaN             : " f"{np.sum(~np.isfinite(data_value_vsw))}")
print(f"Valid           : " f"{np.sum(np.isfinite(data_value_vsw))}")

# FINAL HGBR DATA SUMMARY
print()
print("MODEL HGBR INPUT DATA SUMMARY")
print("Forecast origin : once per day")
print("History         : 3 consecutive days")
print("Forecast        : 3 consecutive days")
print()
print("History VTEC       : 3 days / 15 minutes")
print("History Dst        : 3 days / 1 hour")
print("History F10.7      : 3 days / 1 day")
print("History IMF Bz     : 3 days / 15 minutes")
print("History Vsw/Vtotal : 3 days / 15 minutes")
print()
print("Forecast scheme:")
print("  D1 = first forecast day")
print("  D2 = second forecast day")
print("  D3 = third forecast day")
print()
print("Example:")
print("  Input : 1, 2, 3")
print("  D1    : 4")
print("  D2    : 5")
print("  D3    : 6")
print()
print("VTEC variables:")
print("  data_time_tec")
print("  data_value_tec")
print()
print("Dst variables:")
print("  data_time_dst")
print("  data_value_dst")
print()
print("F10.7 variables:")
print("  data_time_f107")
print("  data_value_f107")
print()
print("IMF Bz variables:")
print("  data_time_bz")
print("  data_value_bz")
print()
print("Vsw/Vtotal variables:")
print("  data_time_vsw")
print("  data_value_vsw")

VTEC DATA - BEFORE DUPLICATE HANDLING
File       : dataset_learning_VTEC-15-min_ptn.txt
Total data : 128544
Start time : 2023-01-02T00:00:00
End time   : 2026-09-30T23:45:00
NaN        : 3983
Duplicate timestamp : 0
VTEC unique time interval:
900 seconds
VTEC time interval check : OK (15 minutes)

VTEC DATA - FINAL
Total timestamp : 131328
Start time      : 2023-01-02T00:00:00
End time        : 2026-09-30T23:45:00
NaN             : 6767
Valid           : 124561
DST DATA - BEFORE DUPLICATE HANDLING
File       : dataset_learning_dst-index.txt
Total data : 32842
Start time : 2023-01-01T00:00:00
End time   : 2026-09-30T23:00:00
NaN        : 0
Duplicate timestamp : 0

Dst unique time interval:
3600 seconds
Dst time interval check : OK (1 hour)

DST DATA - FINAL
Total timestamp : 32856
Start time      : 2023-01-01T00:00:00
End time        : 2026-09-30T23:00:00
NaN             : 14
Valid           : 32842
F10.7 DATA - BEFORE DUPLICATE HANDLING
File       : dataset_learning_F10-7.txt
Total dat

In [3]:

# ============================================================
# CELL 3 — BUILD DAILY FORECAST DATASET — HGBR
# ============================================================

import datetime as dt
import numpy as np
import pandas as pd


# 1. CONFIGURATION
RESOLUTION_MINUTES = 15
INPUT_DAYS = 3
FORECAST_DAYS = 3
SAMPLES_PER_DAY = (24 * 60 // RESOLUTION_MINUTES)

# VTEC
INPUT_STEPS = (INPUT_DAYS * SAMPLES_PER_DAY)

# Dst
DST_SAMPLES_PER_DAY = 24
DST_INPUT_STEPS = (INPUT_DAYS * DST_SAMPLES_PER_DAY)

# F10.7
F107_INPUT_STEPS = INPUT_DAYS

# IMF Bz
BZ_INPUT_STEPS = (INPUT_DAYS * SAMPLES_PER_DAY)

# Vsw / Vtotal
VSW_INPUT_STEPS = (INPUT_DAYS * SAMPLES_PER_DAY)

# Forecast
FORECAST_STEPS = (FORECAST_DAYS * SAMPLES_PER_DAY)

# DAILY FORECAST ORIGIN
ORIGIN_STEP = SAMPLES_PER_DAY

UTC_OFFSET_HOURS = 7

# 2. TRAIN / VALIDATION / TEST PERIOD
TRAIN_START = dt.datetime(2023, 1, 1, 0, 0)
TRAIN_END = dt.datetime(2024, 12, 31, 23, 59)

VAL_START = dt.datetime(2025, 1, 1, 0, 0)
VAL_END = dt.datetime(2025, 12, 31, 23, 59)

TEST_START = dt.datetime(2026, 1, 1, 0, 0)
TEST_END = dt.datetime(2026, 9, 30, 23, 59)

# 3. FEATURE CONFIGURATION
DOY_FEATURES = 6
TOTAL_FEATURES = (
    INPUT_STEPS
    + DST_INPUT_STEPS
    + F107_INPUT_STEPS
    + BZ_INPUT_STEPS
    + VSW_INPUT_STEPS
    + DOY_FEATURES)

# 4. DISPLAY CONFIGURATION
print("BUILD DAILY FORECAST DATASET — HGBR VTEC")
print("VTEC resolution       :", RESOLUTION_MINUTES, "minutes")
print("Input duration        :", INPUT_DAYS, "days")
print("Forecast horizon      :", FORECAST_DAYS, "days")
print("Samples per day       :", SAMPLES_PER_DAY)
print("Input VTEC steps      :", INPUT_STEPS)
print("Input Dst steps       :", DST_INPUT_STEPS)
print("Input F10.7 steps     :", F107_INPUT_STEPS)
print("Input IMF Bz steps    :", BZ_INPUT_STEPS)
print("Input Vsw/Vtotal steps:", VSW_INPUT_STEPS)
print("Forecast steps        :", FORECAST_STEPS)
print("Forecast origin       : once per day")
print("Origin step           :", ORIGIN_STEP, "samples")
print("UTC offset            : UTC+", UTC_OFFSET_HOURS)
print()
print("Training period   :", TRAIN_START, "->", TRAIN_END)
print("Validation period :", VAL_START, "->", VAL_END)
print("Testing period    :", TEST_START, "->", TEST_END)
print()
print("Total input features:", TOTAL_FEATURES)

# 5. CHECK CELL 2 OUTPUT
required_variables = [
    "data_time_tec",
    "data_value_tec",
    "data_time_dst",
    "data_value_dst",
    "data_time_f107",
    "data_value_f107",
    "data_time_bz",
    "data_value_bz",
    "data_time_vsw",
    "data_value_vsw",]

for variable_name in required_variables:
    if variable_name not in globals():
        raise NameError(f"Variabel '{variable_name}' tidak ditemukan. "
                        "Jalankan CELL 2 terlebih dahulu.")

# 6. CHECK DATA LENGTH
for variable_name in required_variables:
    if len(globals()[variable_name]) == 0:
        raise ValueError(f"Variabel '{variable_name}' kosong.")

# 7. CHECK TIME / VALUE LENGTH
data_pairs = [
    ("data_time_tec", "data_value_tec"),
    ("data_time_dst", "data_value_dst"),
    ("data_time_f107", "data_value_f107"),
    ("data_time_bz", "data_value_bz"),
    ("data_time_vsw", "data_value_vsw"),]

for time_name, value_name in data_pairs:
    if len(globals()[time_name]) != len(globals()[value_name]):
        raise ValueError(f"Panjang {time_name} dan {value_name} berbeda.")

# 8. PREPARE CONTAINERS
X_train = []
y_train = []

X_val = []
y_val = []

X_test = []
y_test = []

timestamps_train = []
timestamps_val = []
timestamps_test = []

forecast_doy_train = []
forecast_doy_val = []
forecast_doy_test = []

nan_X_train = []
nan_y_train = []

nan_X_val = []
nan_y_val = []

nan_X_test = []
nan_y_test = []

# 9. CALCULATE NUMBER OF DAILY BLOCKS
last_input_start_idx = (len(data_value_tec) - INPUT_STEPS - FORECAST_STEPS)

print()
print("BUILDING DAILY FORECAST BLOCKS")
print("Available VTEC points :", len(data_value_tec))
print("Available Dst points  :", len(data_value_dst))
print("Available F10.7 days  :", len(data_value_f107))
print("Available IMF Bz pts  :", len(data_value_bz))
print("Available Vsw points  :", len(data_value_vsw))
print("Maximum start index   :", last_input_start_idx)

# 10. BUILD DAILY BLOCK FORECAST DATASET
block_counter = 0
for input_start_idx in range(0, last_input_start_idx + 1, ORIGIN_STEP):
    # Input VTEC dan target VTEC
    input_end_idx = (input_start_idx + INPUT_STEPS)
    forecast_end_idx = (input_end_idx + FORECAST_STEPS)
    input_vtec = data_value_tec[input_start_idx:input_end_idx]
    target_vtec = data_value_tec[input_end_idx:forecast_end_idx]

    if len(input_vtec) != INPUT_STEPS:
        continue

    if len(target_vtec) != FORECAST_STEPS:
        continue

    # Time information

    input_start_time = pd.Timestamp(data_time_tec[input_start_idx]).to_pydatetime()
    input_end_time = pd.Timestamp(data_time_tec[input_end_idx - 1]).to_pydatetime()
    forecast_start_time = pd.Timestamp(data_time_tec[input_end_idx]).to_pydatetime()
    forecast_end_time = pd.Timestamp(data_time_tec[forecast_end_idx - 1]).to_pydatetime()

    # Historical Dst
    # 72 hourly values sebelum forecast origin.

    dst_origin_time = pd.Timestamp(forecast_start_time)
    dst_start_time = (dst_origin_time - pd.Timedelta(hours=INPUT_DAYS * 24))
    dst_end_time = (dst_origin_time - pd.Timedelta(hours=1))
    dst_mask = ((data_time_dst >= dst_start_time) & (data_time_dst <= dst_end_time))
    input_dst = np.asarray(data_value_dst[dst_mask], dtype=np.float32)

    if len(input_dst) != DST_INPUT_STEPS:
        continue

    # Historical F10.7
    # Menggunakan 3 nilai harian sebelum forecast origin.

    forecast_origin_date = (pd.Timestamp(forecast_start_time).normalize())
    f107_dates = pd.date_range(start=(forecast_origin_date - pd.Timedelta(days=INPUT_DAYS)), 
                               periods=F107_INPUT_STEPS, freq="D")

    input_f107 = []
    for f107_date in f107_dates:
        matching_index = np.where(data_time_f107 == np.datetime64(f107_date.date(), "D"))[0]
        if len(matching_index) == 0:
            input_f107.append(np.nan)

        else:
            input_f107.append(data_value_f107[matching_index[0]])

    input_f107 = np.asarray(input_f107, dtype=np.float32)
    if len(input_f107) != F107_INPUT_STEPS:
        continue


    # Historical IMF Bz
    # 288 nilai 15-menit sebelum forecast origin.
    bz_origin_time = pd.Timestamp(forecast_start_time)
    bz_start_time = (bz_origin_time - pd.Timedelta(minutes=INPUT_DAYS * 24 * 60))
    bz_end_time = (bz_origin_time - pd.Timedelta(minutes=RESOLUTION_MINUTES))
    bz_mask = ((data_time_bz >= bz_start_time) & (data_time_bz <= bz_end_time))
    input_bz = np.asarray(data_value_bz[bz_mask], dtype=np.float32)

    if len(input_bz) != BZ_INPUT_STEPS:
        continue

    # Historical Vsw / Vtotal
    # 288 nilai 15-menit sebelum forecast origin.
    vsw_origin_time = pd.Timestamp(forecast_start_time)
    vsw_start_time = (vsw_origin_time - pd.Timedelta(minutes=INPUT_DAYS * 24 * 60))
    vsw_end_time = (vsw_origin_time - pd.Timedelta(minutes=RESOLUTION_MINUTES))
    vsw_mask = ((data_time_vsw >= vsw_start_time) & (data_time_vsw <= vsw_end_time))
    input_vsw = np.asarray(data_value_vsw[vsw_mask], dtype=np.float32)

    if len(input_vsw) != VSW_INPUT_STEPS:
        continue

    # Forecast DOY
    # D1, D2, D3 menggunakan waktu lokal UTC+7.

    doy_features = []
    doy_info = []
    for day_idx in range(FORECAST_DAYS):
        forecast_day_idx = (input_end_idx + day_idx * SAMPLES_PER_DAY)
        if forecast_day_idx >= len(data_time_tec):
            break

        forecast_day_utc = pd.Timestamp(data_time_tec[forecast_day_idx]).to_pydatetime()
        forecast_day_local = (forecast_day_utc + dt.timedelta(hours=UTC_OFFSET_HOURS))
        doy = forecast_day_local.timetuple().tm_yday
        angle = ( 2.0 * np.pi * doy / 365.25)

        doy_sin = np.sin(angle)
        doy_cos = np.cos(angle)
        doy_features.extend([doy_sin, doy_cos])

        doy_info.append({
            "day": day_idx + 1,
            "utc": forecast_day_utc,
            "local": forecast_day_local,
            "doy": doy,
            "sin": doy_sin,
            "cos": doy_cos})

    if len(doy_features) != DOY_FEATURES:
        continue

    # Combine features
    #
    # 1. Historical VTEC       : 288
    # 2. Historical Dst        : 72
    # 3. Historical F10.7      : 3
    # 4. Historical IMF Bz     : 288
    # 5. Historical Vsw/Vtotal : 288
    # 6. Forecast DOY          : 6
    #
    # Total = 945 features

    features = np.concatenate([
        np.asarray(input_vtec, dtype=np.float32),
        np.asarray(input_dst, dtype=np.float32),
        np.asarray(input_f107, dtype=np.float32),
        np.asarray(input_bz, dtype=np.float32),
        np.asarray(input_vsw, dtype=np.float32),
        np.asarray(doy_features, dtype=np.float32)])

    target = np.asarray(target_vtec, dtype=np.float32)

    # Check feature and target size
    if len(features) != TOTAL_FEATURES:
        continue

    if len(target) != FORECAST_STEPS:
        continue

    # NaN statistics
    feature_nan_count = np.isnan(features).sum()
    target_nan_count = np.isnan(target).sum()

    # TIMESTAMP INFORMATION
    timestamp_info = {
        "input_start": input_start_time,
        "input_end": input_end_time,
        "forecast_start": forecast_start_time,
        "forecast_end": forecast_end_time}


    # TRAINING BLOCK
    # Seluruh target 3 hari harus berada dalam periode
    # 1 Januari 2023 - 31 Desember 2024.

    if (forecast_start_time >= TRAIN_START and forecast_end_time <= TRAIN_END):
        X_train.append(features)
        y_train.append(target)

        timestamps_train.append(timestamp_info)
        forecast_doy_train.append(doy_info)

        nan_X_train.append(feature_nan_count)
        nan_y_train.append(target_nan_count)

        block_counter += 1

    # VALIDATION BLOCK
    # Seluruh target 3 hari harus berada dalam periode
    # 1 Januari 2025 - 31 Desember 2025.

    elif (forecast_start_time >= VAL_START and forecast_end_time <= VAL_END):
        X_val.append(features)
        y_val.append(target)

        timestamps_val.append(timestamp_info)
        forecast_doy_val.append(doy_info)

        nan_X_val.append(feature_nan_count)
        nan_y_val.append(target_nan_count)

        block_counter += 1

    # TEST BLOCK
    # Seluruh target 3 hari harus berada dalam periode

    elif (forecast_start_time >= TEST_START and forecast_end_time <= TEST_END):

        X_test.append(features)
        y_test.append(target)

        timestamps_test.append(timestamp_info)
        forecast_doy_test.append(doy_info)

        nan_X_test.append(feature_nan_count)
        nan_y_test.append(target_nan_count)

        block_counter += 1

# 11. CONVERT TO NUMPY ARRAY
X_train = np.asarray(X_train, dtype=np.float32).reshape(-1, TOTAL_FEATURES)
y_train = np.asarray(y_train, dtype=np.float32).reshape(-1, FORECAST_STEPS)

X_val = np.asarray(X_val, dtype=np.float32).reshape(-1, TOTAL_FEATURES)
y_val = np.asarray(y_val, dtype=np.float32).reshape(-1, FORECAST_STEPS)

X_test = np.asarray(X_test, dtype=np.float32).reshape(-1, TOTAL_FEATURES)
y_test = np.asarray(y_test, dtype=np.float32).reshape(-1, FORECAST_STEPS)

# 12. DATASET INFORMATION
print()
print("DATASET INFORMATION — HGBR")

print("X_train shape :", X_train.shape)
print("y_train shape :", y_train.shape)

print("X_val shape   :", X_val.shape)
print("y_val shape   :", y_val.shape)

print("X_test shape  :", X_test.shape)
print("y_test shape  :", y_test.shape)

# 13. NAN INFORMATION
print()
print("NAN INFORMATION")

for name, X, y in [
    ("TRAIN", X_train, y_train),
    ("VALIDATION", X_val, y_val),
    ("TEST", X_test, y_test)]:

    print()
    print(name)

    print("  NaN in X:", f"{np.isnan(X).sum():,}")
    print("  NaN in y:", f"{np.isnan(y).sum():,}")

# 14. BLOCK INFORMATION
print()
print("BLOCK INFORMATION")

print("Training blocks  :", len(X_train))
print("Validation blocks:", len(X_val))
print("Testing blocks   :", len(X_test))

print("Total blocks     :", (
    len(X_train)
    + len(X_val)
    + len(X_test)))

# 15. FORECAST DOY INFORMATION
def show_forecast_doy(title, doy_data):
    print()
    print(title)

    if len(doy_data) == 0:
        print("No sample available.")
        return

    for item in doy_data[0]:
        print(
            "D{}".format(item["day"]),
            "| UTC =", item["utc"],
            "| Local =", item["local"],
            "| DOY =", item["doy"],
            "| sin = {:.6f}".format(item["sin"]),
            "| cos = {:.6f}".format(item["cos"]))


print()
print("FORECAST DOY INFORMATION")

show_forecast_doy("First training block:", forecast_doy_train)
show_forecast_doy("First validation block:", forecast_doy_val)
show_forecast_doy("First testing block:", forecast_doy_test)

# 16. FIRST SAMPLE CHECK

print()
print("FIRST SAMPLE CHECK")
feature_sections = {
    "VTEC": (0, INPUT_STEPS),
    "Dst": (INPUT_STEPS, INPUT_STEPS + DST_INPUT_STEPS),
    "F10.7": (INPUT_STEPS + DST_INPUT_STEPS, INPUT_STEPS + DST_INPUT_STEPS + F107_INPUT_STEPS),
    "IMF Bz": (INPUT_STEPS + DST_INPUT_STEPS + F107_INPUT_STEPS,
               INPUT_STEPS + DST_INPUT_STEPS + F107_INPUT_STEPS + BZ_INPUT_STEPS),
    "Vsw": (INPUT_STEPS + DST_INPUT_STEPS + F107_INPUT_STEPS + BZ_INPUT_STEPS,
            INPUT_STEPS + DST_INPUT_STEPS + F107_INPUT_STEPS + BZ_INPUT_STEPS + VSW_INPUT_STEPS),
    "DOY": (INPUT_STEPS + DST_INPUT_STEPS + F107_INPUT_STEPS + BZ_INPUT_STEPS + VSW_INPUT_STEPS,
            TOTAL_FEATURES)}

for name, X, y in [
    ("TRAIN", X_train, y_train),
    ("VALIDATION", X_val, y_val),
    ("TEST", X_test, y_test)]:

    print()
    print(name)

    if len(X) == 0:
        print("No sample available.")
        continue

    for feature_name, (start, end) in feature_sections.items():
        print(f"{feature_name:8s} shape:", X[0, start:end].shape, "| NaN:", np.isnan(X[0, start:end]).sum())

    print("Target shape:", y[0].shape, "| NaN:", np.isnan(y[0]).sum())

# 17. FINAL CHECK

print()
print("FINAL CHECK — HGBR Forecast VTEC")

print("Expected X features :", TOTAL_FEATURES)
print("Actual X_train      :", X_train.shape[1])
print("Actual X_val        :", X_val.shape[1])
print("Actual X_test       :", X_test.shape[1])
print("Expected target     :", FORECAST_STEPS)
print("Actual y_train      :", y_train.shape[1])
print("Actual y_val        :", y_val.shape[1])
print("Actual y_test       :", y_test.shape[1])
print()
print("Feature breakdown:")
print("  VTEC       :", INPUT_STEPS)
print("  Dst        :", DST_INPUT_STEPS)
print("  F10.7      :", F107_INPUT_STEPS)
print("  IMF Bz     :", BZ_INPUT_STEPS)
print("  Vsw/Vtotal :", VSW_INPUT_STEPS)
print("  Forecast DOY:", DOY_FEATURES)
print("  Total      :", TOTAL_FEATURES)
print()
print("Forecast scheme:")
print("  Origin : once per day")
print("  Input  : previous 3 days")
print("  D1     : first forecast day")
print("  D2     : second forecast day")
print("  D3     : third forecast day")
print()
print("Dataset split:")
print("  Train      : 2023-2024")
print("  Validation : 2025")
print("  Test       : Jan-Oct 2026")
print()
print("CELL 3 — HGBR Forecast VTEC DONE")


BUILD DAILY FORECAST DATASET — HGBR VTEC
VTEC resolution       : 15 minutes
Input duration        : 3 days
Forecast horizon      : 3 days
Samples per day       : 96
Input VTEC steps      : 288
Input Dst steps       : 72
Input F10.7 steps     : 3
Input IMF Bz steps    : 288
Input Vsw/Vtotal steps: 288
Forecast steps        : 288
Forecast origin       : once per day
Origin step           : 96 samples
UTC offset            : UTC+ 7

Training period   : 2023-01-01 00:00:00 -> 2024-12-31 23:59:00
Validation period : 2025-01-01 00:00:00 -> 2025-12-31 23:59:00
Testing period    : 2026-01-01 00:00:00 -> 2026-09-30 23:59:00

Total input features: 945

BUILDING DAILY FORECAST BLOCKS
Available VTEC points : 131328
Available Dst points  : 32856
Available F10.7 days  : 1369
Available IMF Bz pts  : 131424
Available Vsw points  : 131424
Maximum start index   : 130752

DATASET INFORMATION — HGBR
X_train shape : (725, 945)
y_train shape : (725, 288)
X_val shape   : (363, 945)
y_val shape   : (363, 288)

In [4]:

# ============================================================
# CELL 4 — TRAIN HGBR MODELS FOR HGBR VTEC DAILY FORECAST
# 3-DAY FORECASTING
# ============================================================

import datetime as dt
import time
import numpy as np

from sklearn.ensemble import HistGradientBoostingRegressor

# 1. TRAINING MODE
# "validation" : train menggunakan data 2023-2024
# "final"      : train menggunakan data 2023-2025

# Gunakan "validation" selama eksperimen parameter.
# Ubah menjadi "final" setelah parameter terbaik ditentukan.

TRAINING_MODE = "final"

# 2. MODEL CONFIGURATION
MODEL_PARAMS = {
    "learning_rate": 0.01,
    "max_iter": 400,
    "max_depth": 8,
    "min_samples_leaf": 150,
    "l2_regularization": 0.0,
    "random_state": 42}

# 3. SELECT TRAINING DATA
if TRAINING_MODE == "validation":
    X_fit = X_train
    y_fit = y_train
    training_period_label = "2023-2024"

elif TRAINING_MODE == "final":
    X_fit = np.concatenate([X_train, X_val], axis=0)
    y_fit = np.concatenate([y_train, y_val], axis=0)
    training_period_label = "2023-2025"

else:
    raise ValueError("TRAINING_MODE harus 'validation' atau 'final'.")

# 4. FEATURE DESIGN
expected_features = (
    INPUT_STEPS
    + DST_INPUT_STEPS
    + F107_INPUT_STEPS
    + BZ_INPUT_STEPS
    + VSW_INPUT_STEPS
    + DOY_FEATURES)

# 5. MODEL INFORMATION
print()
print("HGBR MODEL HGBR VTEC — DAILY 3-DAY FORECAST")

print("Training mode       :", TRAINING_MODE)
print("Training period     :", training_period_label)
print()
print("FEATURE DESIGN")
print(
    "Historical VTEC"
    " + Historical Dst"
    " + Historical F10.7"
    " + Historical IMF Bz"
    " + Historical Vsw/Vtotal"
    " + Forecast DOY D1/D2/D3")

print()
print("MODEL PARAMETERS")
for key, value in MODEL_PARAMS.items():
    print(key, "=", value)

# 6. INPUT FEATURE CONFIGURATION
print()
print("INPUT FEATURE CONFIGURATION")
print("Input days             =", INPUT_DAYS)
print("VTEC input steps       =", INPUT_STEPS)
print("Dst input steps        =", DST_INPUT_STEPS)
print("F10.7 input steps      =", F107_INPUT_STEPS)
print("IMF Bz input steps     =", BZ_INPUT_STEPS)
print("Vsw/Vtotal input steps =", VSW_INPUT_STEPS)
print("Forecast DOY features  =", DOY_FEATURES)
print("Total features         =", expected_features)

# 7. FORECAST CONFIGURATION
print()
print("FORECAST CONFIGURATION")
print("Forecast days           =", FORECAST_DAYS)
print("Forecast steps          =", FORECAST_STEPS)
print("Models per forecast day =", SAMPLES_PER_DAY)
print("Number of HGBR models   =", FORECAST_STEPS)
print("Forecast origin         = once per day")

# 8. FORECAST DAY OUTPUT RANGE
print()
print("FORECAST DAY OUTPUT RANGE")
for forecast_day in range(FORECAST_DAYS):
    output_start = (forecast_day * SAMPLES_PER_DAY)
    output_end = (output_start + SAMPLES_PER_DAY - 1)
    print(f"D{forecast_day + 1} = " f"{output_start} to {output_end}")

# 9. TRAINING DATA INFORMATION
print()
print("TRAINING DATA CONFIGURATION")
print("Training blocks =", X_fit.shape[0])
print("Shape X_fit     =", X_fit.shape)
print("Shape y_fit     =", y_fit.shape)
print("NaN pada X_fit  =", np.isnan(X_fit).sum())
print("NaN pada y_fit  =", np.isnan(y_fit).sum())

# 10. SAFETY CHECK DATASET
if X_fit.ndim != 2:
    raise ValueError("X_fit harus memiliki dimensi 2.")

if y_fit.ndim != 2:
    raise ValueError("y_fit harus memiliki dimensi 2.")

if X_fit.shape[1] != expected_features:
    raise ValueError(
        "Jumlah feature tidak sesuai. "
        f"Expected = {expected_features}, "
        f"Actual = {X_fit.shape[1]}")

if y_fit.shape[1] != FORECAST_STEPS:
    raise ValueError(
        "Jumlah target output tidak sesuai. "
        f"Expected = {FORECAST_STEPS}, "
        f"Actual = {y_fit.shape[1]}")

if X_fit.shape[0] != y_fit.shape[0]:
    raise ValueError("Jumlah sample X_fit dan y_fit berbeda.")

if X_fit.shape[0] == 0:
    raise ValueError("Tidak ada sample training.")

if FORECAST_STEPS != (FORECAST_DAYS * SAMPLES_PER_DAY):
    raise ValueError("FORECAST_STEPS tidak konsisten.")

print()
print("DATASET SAFETY CHECK = PASSED")

# 11. MODEL CONTAINER

# Satu model untuk setiap forecast output step.
# models[0]   -> D1, titik pertama
# models[95]  -> D1, titik terakhir
# models[96]  -> D2, titik pertama
# models[191] -> D2, titik terakhir
# models[192] -> D3, titik pertama
# models[287] -> D3, titik terakhir
# Model yang tidak dapat dilatih akan bernilai None.

models = [None for _ in range(FORECAST_STEPS)]


# 12. TRAINING INFORMATION CONTAINER
training_sample_count = np.zeros(FORECAST_STEPS, dtype=np.int32)
training_target_nan_count = np.zeros(FORECAST_STEPS, dtype=np.int32)
training_time_per_model = np.full(FORECAST_STEPS, np.nan, dtype=np.float64)
training_n_iter = np.full(FORECAST_STEPS, np.nan, dtype=np.float64)

# 13. MINIMUM TRAINING SAMPLE
MIN_TRAIN_SAMPLES = 100

# 14. START TRAINING

print()
print("START TRAINING HGBR MODEL")

training_start_time = time.time()
trained_model_count = 0
skipped_model_count = 0
for output_step in range(FORECAST_STEPS):
    # Forecast day information
    forecast_day_index = (output_step // SAMPLES_PER_DAY)
    step_within_day = (output_step % SAMPLES_PER_DAY)
    forecast_day_number = (forecast_day_index + 1)

    # Target for current output step
    y_current = y_fit[:, output_step]

    # Valid target mask
    # X boleh mengandung NaN.
    # Hanya target y yang harus valid.

    valid_target_mask = np.isfinite(y_current)
    n_valid = int(np.sum(valid_target_mask))
    n_nan = int(np.sum(~valid_target_mask))
    training_sample_count[output_step] = n_valid
    training_target_nan_count[output_step] = n_nan

    # Check minimum training sample
    if n_valid < MIN_TRAIN_SAMPLES:
        skipped_model_count += 1
        print("SKIP MODEL", output_step + 1, "/", FORECAST_STEPS, 
              "| Day = D" + str(forecast_day_number), "| Valid samples =", n_valid)

        continue

    # Select training data
    # NaN pada X tetap dipertahankan.
    X_current = X_fit[valid_target_mask]
    y_current_valid = y_current[valid_target_mask]

    # Create HGBR model
    model_current = HistGradientBoostingRegressor(**MODEL_PARAMS)

    # Train current model
    model_start_time = time.time()
    model_current.fit(X_current, y_current_valid)
    model_end_time = time.time()

    # Training time and iteration
    model_training_seconds = (model_end_time - model_start_time)
    training_time_per_model[output_step] = (model_training_seconds)
    training_n_iter[output_step] = (model_current.n_iter_)

    # Save model
    models[output_step] = model_current
    trained_model_count += 1

    # Progress information
    if (
        output_step == 0
        or (output_step + 1) % 24 == 0
        or step_within_day == SAMPLES_PER_DAY - 1
        or output_step == FORECAST_STEPS - 1):

        forecast_minutes = ((output_step + 1) * RESOLUTION_MINUTES)
        forecast_hours = (forecast_minutes / 60.0)
        elapsed_time = (time.time() - training_start_time)

        print("Model", output_step + 1, "/", FORECAST_STEPS,
              "| Day = D" + str(forecast_day_number), 
              "| Forecast position =", round(forecast_hours, 2), "hours", "| Valid samples =",
              n_valid, "| Model time =", round(model_training_seconds, 2), "seconds", "| Elapsed =", 
              round(elapsed_time, 2), "seconds")


# 15. END TRAINING
training_end_time = time.time()
total_training_seconds = (training_end_time - training_start_time)

# 16. TRAINING SUMMARY
print()
print("TRAINING SUMMARY")

print("Training mode        =", TRAINING_MODE)
print("Training period      =", training_period_label)
print("Requested models     =", FORECAST_STEPS)
print("Trained models       =", trained_model_count)
print("Skipped models       =", skipped_model_count)

print("Minimum valid samples =", training_sample_count.min())
print("Maximum valid samples =", training_sample_count.max())
print("Mean valid samples    =", round(training_sample_count.mean(), 2))

print("Total training time   =", round(total_training_seconds, 2), "seconds")

# 17. TRAINING SAMPLE SUMMARY BY FORECAST DAY
print()
print("TRAINING SAMPLE SUMMARY BY FORECAST DAY")
for forecast_day in range(FORECAST_DAYS):
    day_start = (forecast_day * SAMPLES_PER_DAY)
    day_end = (day_start + SAMPLES_PER_DAY)
    day_sample_count = training_sample_count[day_start:day_end]
    day_models = models[day_start:day_end]
    day_model_count = sum(model is not None for model in day_models)

    print()
    print("D" + str(forecast_day + 1))
    print("Trained models =", day_model_count)

    print("Minimum valid samples =", day_sample_count.min())
    print("Maximum valid samples =", day_sample_count.max())

    print("Mean valid samples =", round(day_sample_count.mean(), 2))

# 18. TRAINING TIME SUMMARY
trained_time_mask = np.isfinite(training_time_per_model)
if np.any(trained_time_mask):
    print()
    print("TRAINING TIME SUMMARY")
    print("Mean training time/model =", round( np.mean( training_time_per_model[trained_time_mask]), 3), "seconds")
    print("Minimum training time/model =", round(np.min(training_time_per_model[trained_time_mask]), 3), "seconds")
    print("Maximum training time/model =", round(np.max(training_time_per_model[trained_time_mask]), 3), "seconds")

# 19. HGBR ITERATION DIAGNOSTIC
trained_iter_mask = np.isfinite(training_n_iter)
if np.any(trained_iter_mask):
    n_iter_used = training_n_iter[trained_iter_mask]

    print()
    print("HGBR ITERATION DIAGNOSTIC")
    print("Number of trained models =", len(n_iter_used))
    print("Minimum n_iter =", int(n_iter_used.min()))
    print("Maximum n_iter =", int(n_iter_used.max()))
    print("Mean n_iter =", round(n_iter_used.mean(), 2))
    print("Median n_iter =", round(float(np.median(n_iter_used)), 2))
    print("Models reaching max_iter =", int(np.sum(n_iter_used == MODEL_PARAMS["max_iter"])))

# 20. MODEL PARAMETER CHECK
trained_models = [
    model
    for model in models
    if model is not None]

print()
print("HGBR PARAMETER CHECK")

if len(trained_models) > 0:
    reference_model = trained_models[0]
    print("learning_rate    =", reference_model.learning_rate)
    print("max_iter         =", reference_model.max_iter)
    print("max_depth        =", reference_model.max_depth)
    print("min_samples_leaf =", reference_model.min_samples_leaf)

else:
    print("Tidak ada model HGBR yang berhasil dilatih.")

# 21. FINAL MODEL CHECK
print()
print("FINAL MODEL CHECK")

print("Length models =", len(models))
print("Number of None models =", sum(model is None for model in models))
print("Number of trained models =", sum(model is not None for model in models))
if len(models) != FORECAST_STEPS:
    raise ValueError("Jumlah model tidak sesuai FORECAST_STEPS.")

if (trained_model_count + skipped_model_count != FORECAST_STEPS):
    raise ValueError("Jumlah trained + skipped model " "tidak sesuai FORECAST_STEPS.")

print()
print("FINAL TRAINING CHECK = PASSED")

# 22. CELL 4 SUMMARY
print()
print("CELL 4 — HGBR TRAINING DONE")

print("Training mode           =", TRAINING_MODE)
print("Training period         =", training_period_label)

print("HGBR models trained     =", trained_model_count)
print("HGBR models skipped     =", skipped_model_count)
print("Total HGBR models       =", len(models))

print("Learning rate           =", MODEL_PARAMS["learning_rate"])
print("Max iterations          =", MODEL_PARAMS["max_iter"])
print("Max depth               =", MODEL_PARAMS["max_depth"])
print("Min samples leaf        =", MODEL_PARAMS["min_samples_leaf"])

print()
print("Persistence baseline = NOT INCLUDED IN CELL 4")
print("Persistence will be evaluated separately.")

print()
print("CELL 4 — HGBR FINAL CHECK PASSED")



HGBR MODEL HGBR VTEC — DAILY 3-DAY FORECAST
Training mode       : final
Training period     : 2023-2025

FEATURE DESIGN
Historical VTEC + Historical Dst + Historical F10.7 + Historical IMF Bz + Historical Vsw/Vtotal + Forecast DOY D1/D2/D3

MODEL PARAMETERS
learning_rate = 0.01
max_iter = 400
max_depth = 8
min_samples_leaf = 150
l2_regularization = 0.0
random_state = 42

INPUT FEATURE CONFIGURATION
Input days             = 3
VTEC input steps       = 288
Dst input steps        = 72
F10.7 input steps      = 3
IMF Bz input steps     = 288
Vsw/Vtotal input steps = 288
Forecast DOY features  = 6
Total features         = 945

FORECAST CONFIGURATION
Forecast days           = 3
Forecast steps          = 288
Models per forecast day = 96
Number of HGBR models   = 288
Forecast origin         = once per day

FORECAST DAY OUTPUT RANGE
D1 = 0 to 95
D2 = 96 to 191
D3 = 192 to 287

TRAINING DATA CONFIGURATION
Training blocks = 1088
Shape X_fit     = (1088, 945)
Shape y_fit     = (1088, 288)
NaN pada 

In [5]:
# ============================================================
# CELL 5 — PREDICT AND EVALUATE
# DAILY 3-DAY FORECASTING
# HGBR ONLY
# ============================================================

import datetime as dt
import numpy as np

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,)

# INFORMATION
print()
print("CELL 5 — HGBR PREDICTION AND EVALUATION")

print()
print("TRAINING MODE =", TRAINING_MODE)

# SELECT EVALUATION DATA
if TRAINING_MODE == "validation":
    X_eval = X_val
    y_eval = y_val
    EVALUATION_NAME = "VALIDATION"
    EVALUATION_PERIOD = "2025"

elif TRAINING_MODE == "final":
    X_eval = X_test
    y_eval = y_test
    EVALUATION_NAME = "TEST"
    EVALUATION_PERIOD = "JANUARY - SEPTEMBER 2026"

else:
    raise ValueError("TRAINING_MODE harus 'validation' atau 'final'.")

# SELECT FIT DATA
if "X_fit" not in globals() or "y_fit" not in globals():
    raise ValueError("X_fit atau y_fit tidak ditemukan. " "Pastikan Cell 4 sudah dijalankan.")

print()
print("DATA CONFIGURATION")
print("Training mode       =", TRAINING_MODE)
print("Training data       =", X_fit.shape)
print("Evaluation data     =", X_eval.shape)
print("Evaluation period   =", EVALUATION_PERIOD)
print("Training targets    =", y_fit.shape)
print("Evaluation targets  =", y_eval.shape)

# FEATURE CONFIGURATION
expected_features = (
    INPUT_STEPS
    + DST_INPUT_STEPS
    + F107_INPUT_STEPS
    + BZ_INPUT_STEPS
    + VSW_INPUT_STEPS
    + DOY_FEATURES)

print()
print("FEATURE CONFIGURATION")
print("Historical VTEC     =", INPUT_STEPS)
print("Historical Dst      =", DST_INPUT_STEPS)
print("Historical F10.7    =", F107_INPUT_STEPS)
print("Historical IMF Bz   =", BZ_INPUT_STEPS)
print("Historical Vsw      =", VSW_INPUT_STEPS)
print("Forecast DOY        =", DOY_FEATURES)
print("Total features      =", expected_features)

# FORECAST CONFIGURATION
print()
print("FORECAST CONFIGURATION")
print("Forecast days       =", FORECAST_DAYS)
print("Forecast steps      =", FORECAST_STEPS)
print("Steps per day       =", SAMPLES_PER_DAY)
print("Number of models    =", len(models))
print("Forecast origin     = once per day")

# SAFETY CHECK
if X_fit.ndim != 2 or X_eval.ndim != 2:
    raise ValueError("X_fit dan X_eval harus memiliki dimensi 2.")

if y_fit.ndim != 2 or y_eval.ndim != 2:
    raise ValueError("y_fit dan y_eval harus memiliki dimensi 2.")

if X_fit.shape[1] != expected_features:
    raise ValueError("Jumlah feature X_fit tidak sesuai." f"Expected = {expected_features}, " f"Actual = {X_fit.shape[1]}")

if X_eval.shape[1] != expected_features:
    raise ValueError("Jumlah feature X_eval tidak sesuai." f"Expected = {expected_features}, " f"Actual = {X_eval.shape[1]}")

if y_fit.shape[1] != FORECAST_STEPS:
    raise ValueError("Jumlah output y_fit tidak sesuai FORECAST_STEPS.")

if y_eval.shape[1] != FORECAST_STEPS:
    raise ValueError("Jumlah output y_eval tidak sesuai FORECAST_STEPS.")

if X_fit.shape[0] != y_fit.shape[0]:
    raise ValueError("Jumlah block X_fit dan y_fit berbeda.")

if X_eval.shape[0] != y_eval.shape[0]:
    raise ValueError("Jumlah block X_eval dan y_eval berbeda.")

if len(models) != FORECAST_STEPS:
    raise ValueError("Jumlah model tidak sesuai FORECAST_STEPS.")


if FORECAST_STEPS != FORECAST_DAYS * SAMPLES_PER_DAY:
    raise ValueError("FORECAST_STEPS tidak konsisten dengan " "FORECAST_DAYS dan SAMPLES_PER_DAY.")

print()
print("PREDICTION DATASET SAFETY CHECK = PASSED")

# CREATE PREDICTION ARRAYS
Y_pred_fit = np.full(y_fit.shape, np.nan, dtype=np.float32)
Y_pred_eval = np.full(y_eval.shape, np.nan, dtype=np.float32)

# START PREDICTION
prediction_start_time = dt.datetime.now()

print()
print("START HGBR PREDICTION")
predicted_output_count = 0
skipped_output_count = 0

# PREDICTION LOOP
for output_step in range(FORECAST_STEPS):
    forecast_day_index = (output_step // SAMPLES_PER_DAY)
    forecast_day_number = forecast_day_index + 1
    step_within_day = (output_step % SAMPLES_PER_DAY)
    model_current = models[output_step]

    # SKIP UNAVAILABLE MODEL
    if model_current is None:
        skipped_output_count += 1
        continue

    # PREDICT FIT DATA
    Y_pred_fit[:, output_step] = (model_current.predict(X_fit))

    # PREDICT EVALUATION DATA
    Y_pred_eval[:, output_step] = (model_current.predict(X_eval))

    predicted_output_count += 1
    # PROGRESS INFORMATION
    if (output_step == 0
        or (output_step + 1) % 24 == 0
        or step_within_day == SAMPLES_PER_DAY - 1
        or output_step == FORECAST_STEPS - 1):

        forecast_hours = ((output_step + 1) * RESOLUTION_MINUTES / 60.0)
        elapsed_time = (dt.datetime.now() - prediction_start_time)

        print("Prediction", output_step + 1, "/", FORECAST_STEPS,
              "| Day = D" + str(forecast_day_number), "| Forecast position =", round(forecast_hours, 2),
              "hours", "| Elapsed =", elapsed_time)

# END PREDICTION
prediction_end_time = dt.datetime.now()
total_prediction_time = (prediction_end_time - prediction_start_time)

print()
print("PREDICTION COMPLETED")
print("Total prediction time =", total_prediction_time)
print("Predicted outputs     =", predicted_output_count)
print("Skipped outputs       =", skipped_output_count)

# PREDICTION SHAPE
print()
print("PREDICTION SHAPE")
print("Shape y_fit       =", y_fit.shape)
print("Shape Y_pred_fit  =", Y_pred_fit.shape)
print("Shape y_eval      =", y_eval.shape)
print("Shape Y_pred_eval =", Y_pred_eval.shape)


# PREDICTION NaN INFORMATION
print()
print("PREDICTION NaN INFORMATION")
print("NaN y_fit       =", np.isnan(y_fit).sum())
print("NaN Y_pred_fit  =", np.isnan(Y_pred_fit).sum())
print("NaN y_eval      =", np.isnan(y_eval).sum())
print("NaN Y_pred_eval =", np.isnan(Y_pred_eval).sum())

# METRIC FUNCTION
# Metrics:
# MAE, RMSE, R, R2
# Only valid observation-prediction pairs are used.
def calculate_metrics(observation, prediction):
    valid_mask = (np.isfinite(observation) & np.isfinite(prediction))
    y_valid = observation[valid_mask]
    y_pred_valid = prediction[valid_mask]
    n_total = observation.size
    n_valid = len(y_valid)
    n_missing = n_total - n_valid
    if n_valid == 0:
        return {
            "n_total": n_total,
            "n_valid": 0,
            "n_missing": n_missing,
            "mae": np.nan,
            "rmse": np.nan,
            "r": np.nan,
            "r2": np.nan}

    # MAE
    mae = mean_absolute_error(y_valid, y_pred_valid)

    # RMSE
    rmse = np.sqrt(mean_squared_error(y_valid, y_pred_valid))

    # PEARSON CORRELATION COEFFICIENT (R)
    if (n_valid >= 2 and np.std(y_valid) > 0 and np.std(y_pred_valid) > 0):
        r = np.corrcoef(y_valid, y_pred_valid)[0, 1]

    else:
        r = np.nan

    # R2
    if n_valid >= 2:
        r2 = r2_score(y_valid, y_pred_valid)

    else:
        r2 = np.nan

    return {
        "n_total": n_total,
        "n_valid": n_valid,
        "n_missing": n_missing,
        "mae": mae,
        "rmse": rmse,
        "r": r,
        "r2": r2}

# OVERALL METRICS
fit_metrics = calculate_metrics(y_fit, Y_pred_fit)
eval_metrics = calculate_metrics(y_eval, Y_pred_eval)


# BACKWARD-COMPATIBLE METRIC VARIABLES
train_mae = fit_metrics["mae"]
train_rmse = fit_metrics["rmse"]
train_r = fit_metrics["r"]
train_r2 = fit_metrics["r2"]

eval_mae = eval_metrics["mae"]
eval_rmse = eval_metrics["rmse"]
eval_r = eval_metrics["r"]
eval_r2 = eval_metrics["r2"]


# VALID PAIR ARRAYS
valid_fit_mask = (np.isfinite(y_fit) & np.isfinite(Y_pred_fit))
valid_eval_mask = (np.isfinite(y_eval) & np.isfinite(Y_pred_eval))

y_fit_valid = y_fit[valid_fit_mask]
y_pred_fit_valid = Y_pred_fit[valid_fit_mask]

y_eval_valid = y_eval[valid_eval_mask]
y_pred_eval_valid = Y_pred_eval[valid_eval_mask]

# METRICS BY FORECAST DAY
day_metrics_fit = []
day_metrics_eval = []

for forecast_day in range(FORECAST_DAYS):
    day_start = (forecast_day * SAMPLES_PER_DAY)
    day_end = (day_start + SAMPLES_PER_DAY)

    # FIT DATA
    observation_fit_day = y_fit[:, day_start:day_end]
    prediction_fit_day = Y_pred_fit[:, day_start:day_end]
    metrics_fit_day = calculate_metrics(observation_fit_day, prediction_fit_day)
    day_metrics_fit.append(metrics_fit_day)

    # EVALUATION DATA
    observation_eval_day = y_eval[:, day_start:day_end]
    prediction_eval_day = Y_pred_eval[:, day_start:day_end]
    metrics_eval_day = calculate_metrics(observation_eval_day, prediction_eval_day)
    day_metrics_eval.append(metrics_eval_day)

# PRINT METRIC FUNCTION
def print_metrics(title, metrics):
    print()
    print(title)
    print("Forecast points =", metrics["n_total"])
    print("Valid pairs     =", metrics["n_valid"])
    print("Missing pairs   =", metrics["n_missing"])

    print("MAE  =", round(metrics["mae"], 3)
          if np.isfinite(metrics["mae"])
          else np.nan)

    print("RMSE =", round(metrics["rmse"], 3)
          if np.isfinite(metrics["rmse"])
          else np.nan)

    print("R    =", round(metrics["r"], 4)
          if np.isfinite(metrics["r"])
          else np.nan)

    print("R2   =", round(metrics["r2"], 4)
          if np.isfinite(metrics["r2"])
          else np.nan)

# PRINT OVERALL RESULTS
print_metrics("OVERALL FIT DATA RESULT", fit_metrics)
print_metrics("OVERALL " + EVALUATION_NAME + " RESULT", eval_metrics)

# PRINT RESULTS BY FORECAST DAY
print()
print(EVALUATION_NAME + " PERFORMANCE BY FORECAST DAY")

for forecast_day in range(FORECAST_DAYS):
    print_metrics("D" + str(forecast_day + 1) + " " + EVALUATION_NAME + " RESULT", day_metrics_eval[forecast_day])

print()
print("FIT DATA PERFORMANCE BY FORECAST DAY")
for forecast_day in range(FORECAST_DAYS):
    print_metrics("D" + str(forecast_day + 1) + " FIT RESULT", day_metrics_fit[forecast_day])

# GENERALIZATION GAP
mae_gap = eval_mae - train_mae
rmse_gap = eval_rmse - train_rmse
r_gap = train_r - eval_r
r2_gap = train_r2 - eval_r2

print()
print("OVERALL FIT-" + EVALUATION_NAME + " PERFORMANCE GAP")
print("MAE gap  =", round(mae_gap, 3)
      if np.isfinite(mae_gap)
      else np.nan)

print("RMSE gap =", round(rmse_gap, 3)
      if np.isfinite(rmse_gap)
      else np.nan)

print("R gap    =", round(r_gap, 4)
      if np.isfinite(r_gap)
      else np.nan)

print("R2 gap   =", round(r2_gap, 4)
      if np.isfinite(r2_gap)
      else np.nan)

# GENERALIZATION GAP BY FORECAST DAY
print()
print("FIT-" + EVALUATION_NAME + " GAP BY FORECAST DAY")
for forecast_day in range(FORECAST_DAYS):
    fit_day = day_metrics_fit[forecast_day]
    eval_day = day_metrics_eval[forecast_day]

    day_mae_gap = (eval_day["mae"] - fit_day["mae"])
    day_rmse_gap = (eval_day["rmse"] - fit_day["rmse"])
    day_r_gap = (fit_day["r"] - eval_day["r"])
    day_r2_gap = (fit_day["r2"] - eval_day["r2"])

    print()
    print("D" + str(forecast_day + 1))
    print("MAE gap  =", round(day_mae_gap, 3)
          if np.isfinite(day_mae_gap)
          else np.nan)

    print("RMSE gap =", round(day_rmse_gap, 3)
          if np.isfinite(day_rmse_gap)
          else np.nan)

    print("R gap    =", round(day_r_gap, 4)
          if np.isfinite(day_r_gap)
          else np.nan)

    print("R2 gap   =", round(day_r2_gap, 4)
          if np.isfinite(day_r2_gap)
          else np.nan)

# EVALUATION PREDICTION VALUE INFORMATION
print()
print(EVALUATION_NAME + " PREDICTION VALUE INFORMATION")

if len(y_eval_valid) > 0:
    print("Minimum observation =", np.min(y_eval_valid))
    print("Maximum observation =", np.max(y_eval_valid))
    print("Mean observation    =", np.mean(y_eval_valid))
    print("Minimum prediction  =", np.min(y_pred_eval_valid))
    print("Maximum prediction  =", np.max(y_pred_eval_valid))
    print("Mean prediction     =", np.mean(y_pred_eval_valid))


# FINAL PREDICTION CHECK
print()
print("FINAL PREDICTION CHECK")

if Y_pred_fit.shape != y_fit.shape:
    raise ValueError("Shape Y_pred_fit tidak sesuai y_fit.")

if Y_pred_eval.shape != y_eval.shape:
    raise ValueError("Shape Y_pred_eval tidak sesuai y_eval.")

if (predicted_output_count + skipped_output_count != FORECAST_STEPS):
    raise ValueError("Jumlah predicted + skipped output " "tidak sesuai FORECAST_STEPS.")

print("Predicted outputs =", predicted_output_count)
print("Skipped outputs   =", skipped_output_count)
print("Overall valid fit pairs =", fit_metrics["n_valid"])
print("Overall valid evaluation pairs =", eval_metrics["n_valid"])

# HGBR PARAMETER CONFIRMATION
# ============================================================
print()
print("HGBR PARAMETER CONFIRMATION")
if "MODEL_PARAMS" in globals():
    for key, value in MODEL_PARAMS.items():
        print(key, "=", value)

# FINAL INFORMATION
# ============================================================
print()
print("CELL 5 — M6.0 HGBR PREDICTION AND EVALUATION DONE")
print("Training mode =", TRAINING_MODE)
print("Evaluation    =", EVALUATION_NAME)
print("Metrics       = MAE, RMSE, R, R2")
print("PERSISTENCE   = NOT INCLUDED")


CELL 5 — HGBR PREDICTION AND EVALUATION

TRAINING MODE = final

DATA CONFIGURATION
Training mode       = final
Training data       = (1088, 945)
Evaluation data     = (271, 945)
Evaluation period   = JANUARY - SEPTEMBER 2026
Training targets    = (1088, 288)
Evaluation targets  = (271, 288)

FEATURE CONFIGURATION
Historical VTEC     = 288
Historical Dst      = 72
Historical F10.7    = 3
Historical IMF Bz   = 288
Historical Vsw      = 288
Forecast DOY        = 6
Total features      = 945

FORECAST CONFIGURATION
Forecast days       = 3
Forecast steps      = 288
Steps per day       = 96
Number of models    = 288
Forecast origin     = once per day

PREDICTION DATASET SAFETY CHECK = PASSED

START HGBR PREDICTION
Prediction 1 / 288 | Day = D1 | Forecast position = 0.25 hours | Elapsed = 0:00:00.028126
Prediction 24 / 288 | Day = D1 | Forecast position = 6.0 hours | Elapsed = 0:00:00.438067
Prediction 48 / 288 | Day = D1 | Forecast position = 12.0 hours | Elapsed = 0:00:00.878317
Prediction 

In [6]:

# ============================================================
# CELL 6 — M6.0
# SAVE MODEL + PREDICTION RESULTS + METADATA
# + TRAINING INFORMATION + METRICS + TXT RESULTS
# ============================================================

# 1. CHECK REQUIRED VARIABLES
required_variables = [
    "sourcepath",
    "TRAINING_MODE",
    "X_fit",
    "y_fit",
    "X_eval",
    "y_eval",
    "Y_pred_fit",
    "Y_pred_eval",
    "models",
    "MODEL_PARAMS",
    "MIN_TRAIN_SAMPLES",
    "fit_metrics",
    "eval_metrics",
    "day_metrics_fit",
    "day_metrics_eval",
    "timestamps_train",
    "timestamps_val",
    "timestamps_test",
    "FORECAST_STEPS",
    "FORECAST_DAYS",
    "INPUT_DAYS",
    "INPUT_STEPS",
    "RESOLUTION_MINUTES",
    "SAMPLES_PER_DAY",
    "DST_INPUT_STEPS",
    "F107_INPUT_STEPS",
    "BZ_INPUT_STEPS",
    "VSW_INPUT_STEPS",
    "DOY_FEATURES",]

missing_variables = [name for name in required_variables if name not in globals()]
if missing_variables: 
    raise NameError("Variabel berikut belum tersedia. " "Pastikan Cell 3, Cell 4, dan Cell 5 "
                    "sudah dijalankan:\n" + ", ".join(missing_variables))

# 2. TRAINING MODE INFORMATION
TRAINING_MODE = str(TRAINING_MODE).lower().strip()
if TRAINING_MODE == "validation":
    EVALUATION_NAME = "VALIDATION"
    FIT_PERIOD = "2023-01-01 to 2024-12-31"
    EVALUATION_PERIOD = "2025-01-01 to 2025-12-31"
    timestamps_fit = np.asarray(timestamps_train, dtype=object)
    timestamps_eval = np.asarray(timestamps_val, dtype=object)
    forecast_doy_fit = np.asarray(forecast_doy_train, dtype=object)
    forecast_doy_eval = np.asarray(forecast_doy_val, dtype=object)

elif TRAINING_MODE == "final":
    EVALUATION_NAME = "TEST"
    FIT_PERIOD = "2023-01-01 to 2025-12-31"
    EVALUATION_PERIOD = "2026-01-01 to 2026-10-31"
    timestamps_fit = np.concatenate([np.asarray(timestamps_train, dtype=object), np.asarray(timestamps_val, dtype=object)])
    timestamps_eval = np.asarray(timestamps_test, dtype=object)
    forecast_doy_fit = np.concatenate([np.asarray(forecast_doy_train, dtype=object), np.asarray(forecast_doy_val, dtype=object)])
    forecast_doy_eval = np.asarray(forecast_doy_test, dtype=object)

else:
    raise ValueError("TRAINING_MODE harus 'validation' atau 'final'.")

# 3. OUTPUT DIRECTORY
output_path = os.path.join(sourcepath,"HGBR_M6-0_Daily_VTEC_Forecast_Output")
os.makedirs(output_path, exist_ok=True)

# 4. EXPERIMENT INFORMATION
EXPERIMENT_NAME = (
    f"HGBR_M6-0_Daily_"
    f"Input{INPUT_DAYS}Day_"
    f"Forecast{FORECAST_DAYS}Day_"
    f"{TRAINING_MODE.upper()}")

FEATURE_DESIGN_NAME = "M6.0"
FEATURE_DESCRIPTION = (
    "Historical VTEC 15-minute, 3 days; "
    "Historical Dst 1-hour, 3 days; "
    "Historical F10.7 daily, 3 days; "
    "Historical IMF Bz 15-minute, 3 days; "
    "Historical Vsw/Vtotal 15-minute, 3 days; "
    "Forecast DOY sin/cos for D1, D2, and D3.")

feature_order_m60 = [
    "Historical VTEC",
    "Historical Dst",
    "Historical F10.7",
    "Historical IMF Bz",
    "Historical Vsw/Vtotal",
    "Forecast DOY D1 sin/cos",
    "Forecast DOY D2 sin/cos",
    "Forecast DOY D3 sin/cos",]

# 5. SHAPE AND CONFIGURATION CHECK
EXPECTED_FEATURES = (
    INPUT_STEPS
    + DST_INPUT_STEPS
    + F107_INPUT_STEPS
    + BZ_INPUT_STEPS
    + VSW_INPUT_STEPS
    + DOY_FEATURES)

if EXPECTED_FEATURES != 945:
    raise ValueError(f"Jumlah fitur M6.0 seharusnya 945, " f"tetapi konfigurasi menghasilkan {EXPECTED_FEATURES}.")

if X_fit.shape[1] != EXPECTED_FEATURES:
    raise ValueError(f"Jumlah fitur X_fit tidak sesuai: " f"{X_fit.shape[1]} != {EXPECTED_FEATURES}")

if X_eval.shape[1] != EXPECTED_FEATURES:
    raise ValueError(f"Jumlah fitur X_eval tidak sesuai: " f"{X_eval.shape[1]} != {EXPECTED_FEATURES}")

if len(models) != FORECAST_STEPS:
    raise ValueError("Jumlah model tidak sama dengan FORECAST_STEPS.")

if FORECAST_STEPS != 288:
    raise ValueError(f"FORECAST_STEPS seharusnya 288, " f"tetapi ditemukan {FORECAST_STEPS}.")

if Y_pred_fit.shape != y_fit.shape:
    raise ValueError("Shape Y_pred_fit tidak sesuai dengan y_fit.")

if Y_pred_eval.shape != y_eval.shape:
    raise ValueError("Shape Y_pred_eval tidak sesuai dengan y_eval.")

if len(timestamps_fit) != len(y_fit):
    raise ValueError("Jumlah timestamps_fit tidak sesuai dengan y_fit.")

if len(timestamps_eval) != len(y_eval):
    raise ValueError("Jumlah timestamps_eval tidak sesuai dengan y_eval.")

if len(day_metrics_fit) != FORECAST_DAYS:
    raise ValueError("Jumlah day_metrics_fit tidak sesuai FORECAST_DAYS.")

if len(day_metrics_eval) != FORECAST_DAYS:
    raise ValueError("Jumlah day_metrics_eval tidak sesuai FORECAST_DAYS.")

# 6. TRAINING INFORMATION
trained_models = [
    model
    for model in models
    if model is not None]

trained_model_count = len(trained_models)
skipped_model_count = (len(models) - trained_model_count)
if trained_model_count == 0:
    raise ValueError("Tidak ada model HGBR yang berhasil dilatih.")

# Ambil informasi dari Cell 4.

if "training_sample_count" not in globals():
    raise NameError("training_sample_count tidak ditemukan dari Cell 4.")

if "training_target_nan_count" not in globals():
    raise NameError("training_target_nan_count tidak ditemukan dari Cell 4.")

if "training_time_per_model" not in globals():
    raise NameError("training_time_per_model tidak ditemukan dari Cell 4.")


# Konversi waktu pelatihan menjadi detik.
def convert_time_to_seconds(value):
    if hasattr(value, "total_seconds"):
        return float(value.total_seconds())

    return float(value)

training_time_seconds = np.array([convert_time_to_seconds(value) for value in training_time_per_model], dtype=float)
if len(training_time_seconds) != FORECAST_STEPS:
    raise ValueError("Jumlah training_time_per_model tidak sesuai " "dengan FORECAST_STEPS.")

total_training_time_seconds = float(np.nansum(training_time_seconds))

# Gunakan total_training_time jika tersedia dan valid.
if "total_training_time" in globals():
    try:
        total_training_time_seconds = (convert_time_to_seconds(total_training_time))
    except (TypeError, ValueError):
        pass

total_training_time = total_training_time_seconds

# 7. HGBR ITERATION INFORMATION
n_iter_used = np.array([model.n_iter_ for model in trained_models], dtype=int)

n_iter_min = int(np.min(n_iter_used))
n_iter_max = int(np.max(n_iter_used))
n_iter_mean = float(np.mean(n_iter_used))
n_iter_median = float(np.median(n_iter_used))
models_reaching_max_iter = int(np.sum(n_iter_used == MODEL_PARAMS["max_iter"]))

# 8. PREDICTION INFORMATION
predicted_output_count = int(sum(model is not None for model in models))
skipped_output_count = (FORECAST_STEPS - predicted_output_count)

if "total_prediction_time" not in globals():
    total_prediction_time = np.nan

# 9. DATA QUALITY INFORMATION
nan_X_fit = int(np.isnan(X_fit).sum())
nan_y_fit = int(np.isnan(y_fit).sum())

nan_X_eval = int(np.isnan(X_eval).sum())
nan_y_eval = int(np.isnan(y_eval).sum())

valid_y_fit = int(np.isfinite(y_fit).sum())
valid_y_eval = int(np.isfinite(y_eval).sum())

# 10. SAVE HGBR MODELS
model_filename = (EXPERIMENT_NAME + "_Models.joblib")
model_file = os.path.join(output_path, model_filename)

joblib.dump(models, model_file)
# 11. METADATA
metadata = {
    "experiment_name": EXPERIMENT_NAME,
    "model": "HistGradientBoostingRegressor",
    "model_version": "M6.0",
    "feature_design": FEATURE_DESIGN_NAME,
    "feature_description": FEATURE_DESCRIPTION,
    "feature_order": feature_order_m60,
    "training_mode": TRAINING_MODE,
    "evaluation_name": EVALUATION_NAME,
    "fit_period": FIT_PERIOD,
    "evaluation_period": EVALUATION_PERIOD,
    "forecast_origin_frequency": "Once per day",
    "resolution_minutes": RESOLUTION_MINUTES,
    "samples_per_day": SAMPLES_PER_DAY,
    "input_days": INPUT_DAYS,
    "forecast_days": FORECAST_DAYS,
    "input_steps": INPUT_STEPS,
    "forecast_steps": FORECAST_STEPS,
    "dst_input_steps": DST_INPUT_STEPS,
    "f107_input_steps": F107_INPUT_STEPS,
    "bz_input_steps": BZ_INPUT_STEPS,
    "vsw_input_steps": VSW_INPUT_STEPS,
    "doy_features": DOY_FEATURES,
    "number_of_features": EXPECTED_FEATURES,
    "feature_count": {
        "VTEC": INPUT_STEPS,
        "Dst": DST_INPUT_STEPS,
        "F10.7": F107_INPUT_STEPS,
        "IMF_Bz": BZ_INPUT_STEPS,
        "Vsw_Vtotal": VSW_INPUT_STEPS,
        "Forecast_DOY": DOY_FEATURES},

    "target_parameter": "VTEC",
    "target_resolution": "15 minutes",
    "number_of_outputs": FORECAST_STEPS,
    "output_structure": {
        "outputs_per_day": SAMPLES_PER_DAY,
        "forecast_days": FORECAST_DAYS,
        "D1_output_range": [0, SAMPLES_PER_DAY - 1],
        "D2_output_range": [SAMPLES_PER_DAY, 2 * SAMPLES_PER_DAY - 1],
        "D3_output_range": [2 * SAMPLES_PER_DAY, 3 * SAMPLES_PER_DAY - 1]},

    "model_parameters": MODEL_PARAMS,
    "minimum_training_samples": MIN_TRAIN_SAMPLES,
    "requested_models": FORECAST_STEPS,
    "trained_models": trained_model_count,
    "skipped_models": skipped_model_count,
    "training_sample_count": training_sample_count,
    "training_target_nan_count": training_target_nan_count,
    "training_time_per_model_seconds": training_time_seconds,
    "total_training_time_seconds": total_training_time_seconds,
    "n_iter_used": n_iter_used,
    "n_iter_min": n_iter_min,
    "n_iter_max": n_iter_max,
    "n_iter_mean": n_iter_mean,
    "n_iter_median": n_iter_median,
    "models_reaching_max_iter": models_reaching_max_iter,
    "predicted_outputs": predicted_output_count,
    "skipped_outputs": skipped_output_count,
    "total_prediction_time": total_prediction_time,
    "fit_metrics": fit_metrics,
    "evaluation_metrics": eval_metrics,
    "fit_metrics_by_day": day_metrics_fit,
    "evaluation_metrics_by_day": day_metrics_eval,
    "performance_gap": {
        "MAE_gap": mae_gap,
        "RMSE_gap": rmse_gap,
        "R_gap": r_gap,
        "R2_gap": r2_gap},

    "data_quality": {
        "fit_total_samples": len(y_fit),
        "evaluation_total_samples": len(y_eval),
        "fit_valid_VTEC": valid_y_fit,
        "evaluation_valid_VTEC": valid_y_eval,
        "fit_missing_VTEC": nan_y_fit,
        "evaluation_missing_VTEC": nan_y_eval,
        "fit_missing_input_values": nan_X_fit,
        "evaluation_missing_input_values": nan_X_eval}}

# 12. SAVE METADATA
metadata_filename = (EXPERIMENT_NAME + "_Metadata.joblib")
metadata_file = os.path.join(output_path, metadata_filename)
joblib.dump(metadata, metadata_file)

# 13. SAVE PREDICTION RESULTS
prediction_filename = (EXPERIMENT_NAME + "_Prediction_Results.npz")
prediction_file = os.path.join(output_path,prediction_filename)

np.savez_compressed(
    prediction_file,
    training_mode=TRAINING_MODE,
    evaluation_name=EVALUATION_NAME,
    y_fit=y_fit,
    Y_pred_fit=Y_pred_fit,
    timestamps_fit=np.asarray(
        timestamps_fit,
        dtype=object),

    forecast_doy_fit=np.asarray(
        forecast_doy_fit,
        dtype=object),

    y_eval=y_eval, Y_pred_eval=Y_pred_eval,

    timestamps_eval=np.asarray(timestamps_eval, dtype=object),
    forecast_doy_eval=np.asarray(forecast_doy_eval, dtype=object),
    nan_X_fit=nan_X_fit,
    nan_y_fit=nan_y_fit,
    nan_X_eval=nan_X_eval,
    nan_y_eval=nan_y_eval)


# 14. SAVE TRAINING INFORMATION
training_info_filename = (EXPERIMENT_NAME + "_Training_Info.npz")
training_info_file = os.path.join(output_path, training_info_filename)

np.savez_compressed(training_info_file,
                    training_mode=TRAINING_MODE,
                    training_sample_count=training_sample_count,
                    training_target_nan_count=training_target_nan_count,
                    training_time_per_model_seconds=training_time_seconds,
                    total_training_time_seconds=total_training_time_seconds,
                    n_iter_used=n_iter_used,
                    trained_model_count=trained_model_count,
                    skipped_model_count=skipped_model_count)


# 15. SAVE METRICS TXT
metrics_filename = (EXPERIMENT_NAME + "_Metrics.txt")
metrics_file = os.path.join(output_path, metrics_filename)

def write_metric_section(fid, title, metrics):
    fid.write(f"\n{title}\n")
    fid.write(f"Forecast points : {metrics['n_total']}\n")
    fid.write(f"Valid pairs     : {metrics['n_valid']}\n")
    fid.write(f"Missing pairs   : {metrics['n_missing']}\n")

    for key in ["mae", "rmse", "r", "r2"]:
        value = metrics.get(key, np.nan)
        fid.write(f"{key.upper():<15}: {value:.6f}\n")


def write_day_metric_section(fid, title, day_metrics):
    fid.write(f"\n{title}\n")
    fid.write("=" * 60 + "\n")
    for i, metrics in enumerate(day_metrics):
        fid.write(f"\nD{i + 1}\n")
        for key in [ "n_total", "n_valid", "n_missing", "mae", "rmse", "r", "r2"]:
            value = metrics.get(key, np.nan)
            if isinstance(value, (int, np.integer)):
                fid.write(f"{key.upper():<15}: {value}\n")
            else:
                fid.write(f"{key.upper():<15}: {value:.6f}\n")


with open(metrics_file, "w", encoding="utf-8") as fid:
    fid.write("HGBR M6.0 DAILY 3-DAY VTEC FORECAST\n")
    fid.write(f"Experiment          : {EXPERIMENT_NAME}\n")
    fid.write(f"Training mode       : {TRAINING_MODE}\n")
    fid.write(f"Fit period          : {FIT_PERIOD}\n")
    fid.write(f"Evaluation period   : {EVALUATION_PERIOD}\n")
    fid.write(f"Feature design      : {FEATURE_DESIGN_NAME}\n")
    fid.write(f"Feature count       : {EXPECTED_FEATURES}\n")
    fid.write(f"Input days          : {INPUT_DAYS}\n")
    fid.write(f"Forecast days       : {FORECAST_DAYS}\n")
    fid.write(f"Forecast steps      : {FORECAST_STEPS}\n")
    fid.write(f"Resolution          : {RESOLUTION_MINUTES} minutes\n")
    fid.write(f"Trained models      : {trained_model_count}\n")
    fid.write(f"Skipped models      : {skipped_model_count}\n")
    fid.write(f"Training time       : {total_training_time_seconds:.3f} seconds\n")
    fid.write(f"Prediction time     : {total_prediction_time}\n")
    fid.write("\nMODEL PARAMETERS\n")
    for key, value in MODEL_PARAMS.items():
        fid.write(f"{key} = {value}\n")

    fid.write("\nHGBR ITERATION INFORMATION\n")
    fid.write(f"Minimum n_iter      : {n_iter_min}\n")
    fid.write(f"Maximum n_iter      : {n_iter_max}\n")
    fid.write(f"Mean n_iter         : {n_iter_mean:.3f}\n")
    fid.write(f"Median n_iter       : {n_iter_median:.3f}\n")
    fid.write(f"Models at max_iter  : {models_reaching_max_iter}\n")
    write_metric_section(fid, "OVERALL FIT PERFORMANCE", fit_metrics)
    write_metric_section(fid, f"OVERALL {EVALUATION_NAME} PERFORMANCE", eval_metrics)
    write_day_metric_section(fid, "FIT PERFORMANCE BY FORECAST DAY", day_metrics_fit)
    write_day_metric_section(fid, f"{EVALUATION_NAME} PERFORMANCE BY FORECAST DAY", day_metrics_eval)
    
    fid.write("\nFIT-EVALUATION PERFORMANCE GAP\n")
    fid.write(f"MAE gap  : {mae_gap:.6f}\n")
    fid.write(f"RMSE gap : {rmse_gap:.6f}\n")
    fid.write(f"R gap    : {r_gap:.6f}\n")
    fid.write(f"R2 gap   : {r2_gap:.6f}\n")

    fid.write("\nDATA QUALITY\n")
    fid.write(f"Fit samples             : {len(y_fit)}\n")
    fid.write(f"Evaluation samples      : {len(y_eval)}\n")
    fid.write(f"Fit missing VTEC        : {nan_y_fit}\n")
    fid.write(f"Evaluation missing VTEC : {nan_y_eval}\n")
    fid.write(f"Fit missing inputs      : {nan_X_fit}\n")
    fid.write(f"Evaluation missing inputs: {nan_X_eval}\n")


# 16. FORECAST TIMESTAMP HELPER
def get_forecast_origin(timestamp_item):
    if isinstance(timestamp_item, dict):
        if "forecast_start" in timestamp_item:
            timestamp_item = timestamp_item["forecast_start"]

        elif "datetime" in timestamp_item:
            timestamp_item = timestamp_item["datetime"]

        else:
            raise KeyError("Dictionary timestamp tidak memiliki " "'forecast_start' atau 'datetime'.")

    return pd.Timestamp(timestamp_item)

# 17. SAVE PREDICTION RESULTS TO TXT
def save_prediction_txt(filename, timestamps, observation, prediction):
    if observation.shape != prediction.shape:
        raise ValueError("Shape observation dan prediction berbeda.")

    if len(timestamps) != observation.shape[0]:
        raise ValueError("Jumlah timestamp tidak sesuai jumlah forecast block.")

    with open(filename, "w", encoding="utf-8") as fid:
        fid.write(
            "FORECAST_ORIGIN "
            "TARGET_TIME "
            "FORECAST_DAY "
            "FORECAST_STEP "
            "OBSERVATION "
            "PREDICTION\n")

        for block_idx in range(observation.shape[0]):
            forecast_origin = get_forecast_origin(timestamps[block_idx])
            
            for output_step in range(observation.shape[1]):
                target_time = (forecast_origin + pd.Timedelta(minutes=(output_step + 1) * RESOLUTION_MINUTES))
                obs = observation[block_idx, output_step]
                pred = prediction[block_idx, output_step]
                forecast_day = (output_step // SAMPLES_PER_DAY) + 1
                forecast_step = output_step + 1
                
                obs_string = (f"{obs:.6f}"
                              if np.isfinite(obs)
                              else "nan")

                pred_string = (f"{pred:.6f}"
                               if np.isfinite(pred)
                               else "nan")

                fid.write(
                    f"{forecast_origin:%Y-%m-%d %H:%M:%S} "
                    f"{target_time:%Y-%m-%d %H:%M:%S} "
                    f"{forecast_day} "
                    f"{forecast_step} "
                    f"{obs_string} "
                    f"{pred_string}\n")


# 18. SAVE FIT AND EVALUATION TXT
fit_txt_filename = (EXPERIMENT_NAME + "_Fit_Results.txt")
eval_txt_filename = (EXPERIMENT_NAME + f"_{EVALUATION_NAME}_Results.txt")

fit_txt_file = os.path.join(output_path, fit_txt_filename)
eval_txt_file = os.path.join(output_path, eval_txt_filename)

save_prediction_txt(fit_txt_file, timestamps_fit, y_fit, Y_pred_fit)
save_prediction_txt(eval_txt_file, timestamps_eval, y_eval, Y_pred_eval)


# 19. CHECK SAVED FILES
saved_files = [model_file, metadata_file, prediction_file, training_info_file, metrics_file, fit_txt_file, eval_txt_file]

print()
print("HGBR Model M6.0 SAVED FILE CHECK")
for saved_file in saved_files:
    exists = os.path.exists(saved_file)
    size_mb = (os.path.getsize(saved_file) / (1024 ** 2)
               if exists
               else np.nan)

    print(f"{os.path.basename(saved_file)}"
          f" | Exists = {exists}"
          f" | Size = {size_mb:.3f} MB")

# 20. FINAL INFORMATION
model_file_size_mb = (os.path.getsize(model_file) / (1024 ** 2))

print()
print("CELL 6 — HGBR Model M6.0 COMPLETED")

print(f"Experiment       : {EXPERIMENT_NAME}")
print(f"Training mode    : {TRAINING_MODE}")
print(f"Evaluation       : {EVALUATION_NAME}")
print(f"Fit samples      : {len(y_fit)}")
print(f"Evaluation samples: {len(y_eval)}")
print(f"Feature count    : {EXPECTED_FEATURES}")
print(f"Trained models   : {trained_model_count}")
print(f"Skipped models   : {skipped_model_count}")
print(f"Training time    : {total_training_time_seconds:.2f} seconds")
print(f"Model size       : {model_file_size_mb:.2f} MB")

print()
print("OUTPUT DIRECTORY:")
print(output_path)

print()
print("FIT METRICS:")
print(f"MAE  = {fit_metrics['mae']:.4f}")
print(f"RMSE = {fit_metrics['rmse']:.4f}")
print(f"R    = {fit_metrics['r']:.4f}")
print(f"R2   = {fit_metrics['r2']:.4f}")

print()
print(f"{EVALUATION_NAME} METRICS:")
print(f"MAE  = {eval_metrics['mae']:.4f}")
print(f"RMSE = {eval_metrics['rmse']:.4f}")
print(f"R    = {eval_metrics['r']:.4f}")
print(f"R2   = {eval_metrics['r2']:.4f}")


HGBR Model M6.0 SAVED FILE CHECK
HGBR_M6-0_Daily_Input3Day_Forecast3Day_FINAL_Models.joblib | Exists = True | Size = 590.291 MB
HGBR_M6-0_Daily_Input3Day_Forecast3Day_FINAL_Metadata.joblib | Exists = True | Size = 0.009 MB
HGBR_M6-0_Daily_Input3Day_Forecast3Day_FINAL_Prediction_Results.npz | Exists = True | Size = 1.854 MB
HGBR_M6-0_Daily_Input3Day_Forecast3Day_FINAL_Training_Info.npz | Exists = True | Size = 0.003 MB
HGBR_M6-0_Daily_Input3Day_Forecast3Day_FINAL_Metrics.txt | Exists = True | Size = 0.003 MB
HGBR_M6-0_Daily_Input3Day_Forecast3Day_FINAL_Fit_Results.txt | Exists = True | Size = 19.809 MB
HGBR_M6-0_Daily_Input3Day_Forecast3Day_FINAL_TEST_Results.txt | Exists = True | Size = 4.955 MB

CELL 6 — HGBR Model M6.0 COMPLETED
Experiment       : HGBR_M6-0_Daily_Input3Day_Forecast3Day_FINAL
Training mode    : final
Evaluation       : TEST
Fit samples      : 1088
Evaluation samples: 271
Feature count    : 945
Trained models   : 288
Skipped models   : 0
Training time    : 2474.61 sec

In [ ]:
#### Temukan hyperparameter yang pas, sebelum menggantinya ke mode 'final' pada cell 4.  

In [ ]:
#### jika sudah model Final, lanjut ke cell selanjutnya

In [7]:

# ============================================================
# CELL 7 - FINAL EVALUATION METRICS: TEST
# Experiment: HGBR M6.0
# Training : TRAIN + VALIDATION (2023-2025)
# Testing  : January-September 2026
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score)

# 1. VALIDASI KONDISI MODEL FINAL

if TRAINING_MODE.lower() != "final":
    raise ValueError("Cell 7 harus dijalankan menggunakan TRAINING_MODE = 'final'. "
                     "Ubah mode pada Cell 4, kemudian jalankan ulang Cell 4-6.")

if len(models) != FORECAST_STEPS:
    raise ValueError(f"Jumlah model tidak sesuai. Ditemukan {len(models)}, "
                     f"seharusnya {FORECAST_STEPS}.")

if X_test.shape[0] != y_test.shape[0]:
    raise ValueError("Jumlah sampel X_test dan y_test tidak sama.")

# 2. MENYIAPKAN TARGET DAN PREDIKSI TEST
Y_true_test = np.asarray(y_test, dtype=float)
Y_pred_test = np.full((X_test.shape[0], FORECAST_STEPS), np.nan, dtype=float)

for step, model in enumerate(models):
    if model is None:
        continue

    try:
        Y_pred_test[:, step] = model.predict(X_test)

    except Exception as e:
        print(f"Warning: Prediksi TEST pada step {step + 1} gagal: {e}")

# 3. VALIDASI DIMENSI DATA
if Y_true_test.shape != Y_pred_test.shape:
    raise ValueError(f"Dimensi TEST tidak sama: "
                     f"Target {Y_true_test.shape}, "
                     f"Prediksi {Y_pred_test.shape}")

if Y_true_test.ndim != 2 or Y_true_test.shape[1] != FORECAST_STEPS:
    raise ValueError(f"Dimensi TEST tidak sesuai dengan "
                     f"FORECAST_STEPS = {FORECAST_STEPS}.")

# 4. FUNGSI PERHITUNGAN METRIK
def calculate_metrics_m60(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float).ravel()
    y_pred = np.asarray(y_pred, dtype=float).ravel()

    valid_mask = np.isfinite(y_true) & np.isfinite(y_pred)
    n_total = len(y_true)
    n_valid = int(np.sum(valid_mask))
    n_missing = n_total - n_valid

    if n_valid == 0:
        return {
            "n_total": n_total,
            "n_valid": n_valid,
            "n_missing": n_missing,
            "MAE": np.nan,
            "RMSE": np.nan,
            "R": np.nan,
            "R2": np.nan}

    actual = y_true[valid_mask]
    predicted = y_pred[valid_mask]

    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))

    if n_valid >= 2 and np.std(actual) > 0 and np.std(predicted) > 0:
        correlation = np.corrcoef(actual, predicted)[0, 1]
    else:
        correlation = np.nan

    if n_valid >= 2 and np.std(actual) > 0:
        r2 = r2_score(actual, predicted)
    else:
        r2 = np.nan

    return {
        "n_total": n_total,
        "n_valid": n_valid,
        "n_missing": n_missing,
        "MAE": mae,
        "RMSE": rmse,
        "R": correlation,
        "R2": r2}


# 5. OVERALL TEST METRICS
overall_test_metrics = calculate_metrics_m60(Y_true_test, Y_pred_test)
overall_metrics_df = pd.DataFrame([overall_test_metrics], index=["TEST"])
overall_metrics_df.index.name = "Dataset"

# 6. TEST METRICS BERDASARKAN HORIZON FORECAST
#    D+1, D+2, D+3

day_metrics_df = {}
for day in range(FORECAST_DAYS):
    start_step = day * SAMPLES_PER_DAY
    end_step = start_step + SAMPLES_PER_DAY

    y_true_day = Y_true_test[:, start_step:end_step]
    y_pred_day = Y_pred_test[:, start_step:end_step]
    day_metrics = calculate_metrics_m60(y_true_day, y_pred_day)

    day_name = f"D+{day + 1}"
    day_metrics_df[day_name] = pd.DataFrame([day_metrics], index=["TEST"])
    day_metrics_df[day_name].index.name = "Dataset"

# 7. MENAMPILKAN HASIL EVALUASI
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

print("HGBR model M6.0 - FINAL OVERALL TEST METRICS")
display(overall_metrics_df)
for day_name, metrics_df in day_metrics_df.items():
    print(f"HGBR model M6.0 - FINAL {day_name} TEST METRICS")
    display(metrics_df)

# 8. RINGKASAN EVALUASI FINAL
print("HGBR model M6.0 - FINAL EVALUATION SUMMARY")

print(f"Training mode       : {TRAINING_MODE}")
print("Training data       : TRAIN + VALIDATION (2023-2025)")
print("Testing period      : January-September 2026")
print(f"Number of models    : {len(models)}")
print(f"Forecast horizon    : {FORECAST_DAYS} days")
print(f"Forecast steps      : {FORECAST_STEPS}")
print(f"Samples per day     : {SAMPLES_PER_DAY}")

valid_count = np.sum(np.isfinite(Y_true_test) & np.isfinite(Y_pred_test))
total_count = Y_true_test.size
print("\nNumber of TEST forecast samples:")
print(f"Forecast origins    : {Y_true_test.shape[0]:,}")
print(f"Total target points: {total_count:,}")
print(f"Valid pairs        : {valid_count:,}")
print(f"Missing pairs      : {total_count - valid_count:,}")
print("\nCell 7 - Final TEST evaluation completed successfully.")


HGBR model M6.0 - FINAL OVERALL TEST METRICS


,n_total,n_valid,n_missing,MAE,RMSE,R,R2
Dataset,,,,,,,
TEST,78048,77347,701,4.6201,5.8573,0.9620,0.9036


HGBR model M6.0 - FINAL D+1 TEST METRICS


,n_total,n_valid,n_missing,MAE,RMSE,R,R2
Dataset,,,,,,,
TEST,26016,25782,234,4.0307,5.2246,0.9692,0.9234


HGBR model M6.0 - FINAL D+2 TEST METRICS


,n_total,n_valid,n_missing,MAE,RMSE,R,R2
Dataset,,,,,,,
TEST,26016,25782,234,4.6451,5.8441,0.9626,0.9040


HGBR model M6.0 - FINAL D+3 TEST METRICS


,n_total,n_valid,n_missing,MAE,RMSE,R,R2
Dataset,,,,,,,
TEST,26016,25783,233,5.1845,6.4399,0.9566,0.8833


HGBR model M6.0 - FINAL EVALUATION SUMMARY
Training mode       : final
Training data       : TRAIN + VALIDATION (2023-2025)
Testing period      : January-September 2026
Number of models    : 288
Forecast horizon    : 3 days
Forecast steps      : 288
Samples per day     : 96

Number of TEST forecast samples:
Forecast origins    : 271
Total target points: 78,048
Valid pairs        : 77,347
Missing pairs      : 701

Cell 7 - Final TEST evaluation completed successfully.


In [8]:

# ============================================================
# CELL 8 - FINAL TEST RESULTS AND METRICS
# Experiment: HGBR M6.0
# Training : TRAIN + VALIDATION (2023-2025)
# Testing  : January-September 2026
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score)

# 1. VALIDASI KONDISI FINAL

if TRAINING_MODE.lower() != "final":
    raise ValueError("Cell 8 hanya dapat dijalankan dengan "
                     "TRAINING_MODE = 'final'.")

if "Y_true_test" not in globals() or "Y_pred_test" not in globals():
    raise NameError("Y_true_test atau Y_pred_test tidak ditemukan. "
                    "Jalankan Cell 7 terlebih dahulu.")

if len(timestamps_test) != Y_true_test.shape[0]:
    raise ValueError("Jumlah timestamps_test tidak sama dengan "
                     "jumlah sampel TEST.")

if Y_true_test.shape != Y_pred_test.shape:
    raise ValueError("Dimensi Y_true_test dan Y_pred_test tidak sama.")

# 2. CALCULATE TEST METRICS
def calculate_test_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float).ravel()
    y_pred = np.asarray(y_pred, dtype=float).ravel()

    valid_mask = (np.isfinite(y_true) & np.isfinite(y_pred))

    y_true_valid = y_true[valid_mask]
    y_pred_valid = y_pred[valid_mask]

    n_total = len(y_true)
    n_valid = len(y_true_valid)
    n_missing = n_total - n_valid

    if n_valid == 0:
        return {
            "N_TOTAL": n_total,
            "N_VALID": n_valid,
            "N_MISSING": n_missing,
            "MAE": np.nan,
            "RMSE": np.nan,
            "R": np.nan,
            "R2": np.nan}

    mae = mean_absolute_error(y_true_valid, y_pred_valid)
    rmse = np.sqrt(mean_squared_error(y_true_valid, y_pred_valid))

    if (n_valid >= 2 and np.std(y_true_valid) > 0 and np.std(y_pred_valid) > 0):
        r = np.corrcoef(y_true_valid, y_pred_valid)[0, 1]
        
    else:
        r = np.nan

    if n_valid >= 2 and np.std(y_true_valid) > 0:
        r2 = r2_score(y_true_valid, y_pred_valid)
        
    else:
        r2 = np.nan

    return {
        "N_TOTAL": n_total,
        "N_VALID": n_valid,
        "N_MISSING": n_missing,
        "MAE": mae,
        "RMSE": rmse,
        "R": r,
        "R2": r2}

# 3. OVERALL TEST METRICS
test_metrics = calculate_test_metrics(Y_true_test, Y_pred_test)

# 4. TEST METRICS BY FORECAST DAY
#    D+1, D+2, D+3

day_metrics_test = {}
for day in range(FORECAST_DAYS):
    start = day * SAMPLES_PER_DAY
    end = start + SAMPLES_PER_DAY
    day_name = f"D+{day + 1}"
    day_metrics_test[day_name] = (calculate_test_metrics( Y_true_test[:, start:end], Y_pred_test[:, start:end]))

# 5. DISPLAY TEST METRICS
metrics_rows = []
metrics_rows.append({"PERIOD": "Overall",**test_metrics})
for day, metrics in day_metrics_test.items():
    metrics_rows.append({"PERIOD": day, **metrics})

test_metrics_df = pd.DataFrame(metrics_rows)

print("HGBR model M6.0 - FINAL TEST METRICS")

display(test_metrics_df)
# 6. CREATE TEST RESULTS DATAFRAME

# Forecast start merupakan waktu awal target.
# Timestamp target pertama = forecast_start.
forecast_origins = pd.DatetimeIndex([item["forecast_start"] for item in timestamps_test])

# Nomor langkah prediksi: 1 sampai 288
step_numbers = np.arange(1, FORECAST_STEPS + 1)

# Target time untuk setiap langkah prediksi.
# Langkah pertama dimulai tepat pada forecast_start.
time_offsets = pd.to_timedelta((step_numbers - 1) * RESOLUTION_MINUTES, unit="min")

target_times = (forecast_origins.to_numpy()[:, None] + time_offsets.to_numpy()[None, :])

# Klasifikasi horizon forecast
forecast_days = ((step_numbers - 1) // SAMPLES_PER_DAY + 1)

# Membuat DataFrame hasil prediksi
test_results_df = pd.DataFrame({"FORECAST_ORIGIN": np.repeat(forecast_origins.to_numpy(),FORECAST_STEPS), 
                                "TARGET_TIME": target_times.ravel(), 
                                "FORECAST_DAY": np.tile(forecast_days, len(forecast_origins)),
                                "FORECAST_STEP": np.tile(step_numbers, len(forecast_origins)),
                                "OBSERVATION": Y_true_test.ravel(), "PREDICTION": Y_pred_test.ravel()})

print("HGBR model M6.0 - FINAL TEST RESULTS")

print("Number of forecast origins:", len(forecast_origins))
print("Number of forecast steps :", FORECAST_STEPS)
print("DataFrame shape           :", test_results_df.shape)
display(test_results_df.head(10))

# 7. VALIDASI TIMESTAMP
print("\nTimestamp verification:")
print("First forecast origin :", forecast_origins[0])
print("First target time     :", pd.Timestamp(target_times[0, 0]))
print("Last target time      :", pd.Timestamp(target_times[0, -1]))

# Target pertama harus sama dengan forecast_start
if not np.array_equal(target_times[:, 0], forecast_origins.to_numpy()):
    raise ValueError("Timestamp target pertama tidak sesuai "
                     "dengan forecast_start.")

# 8. SAVE FINAL TEST RESULTS
test_output_dir = output_path #(r'C:\ML\Data_Learn\HGBR_M6-0_Daily_VTEC_Forecast_Output')

# Membuat folder jika belum tersedia
os.makedirs(test_output_dir,exist_ok=True)

# Simpan TXT (tab-separated)
txt_path = os.path.join(test_output_dir, "FINAL_TEST_Results.txt")
test_results_df.to_csv(txt_path, sep="\t", index=False, float_format="%.4f", date_format="%Y-%m-%d %H:%M:%S")

# Simpan CSV
csv_path = os.path.join(test_output_dir, "FINAL_TEST_Results.csv")

test_results_df.to_csv(csv_path, index=False, float_format="%.4f", date_format="%Y-%m-%d %H:%M:%S")

print("\nFinal TEST result files saved:")
print("TXT:", txt_path)
print("CSV:", csv_path)

# 9. SAVE FINAL TEST METRICS
metrics_txt_path = os.path.join(test_output_dir, "FINAL_TEST_Metrics.txt")
test_metrics_df.to_csv(metrics_txt_path, sep="\t", index=False, float_format="%.4f")
metrics_csv_path = os.path.join(test_output_dir, "FINAL_TEST_Metrics.csv")
test_metrics_df.to_csv(metrics_csv_path, index=False, float_format="%.4f")

print("\nFinal TEST metrics saved:")
print("TXT:", metrics_txt_path)
print("CSV:", metrics_csv_path)

# 10. FINAL SUMMARY

print("HGBR model M6.0 - FINAL EXPORT SUMMARY")

print("Training mode       :", TRAINING_MODE)
print("Training period     : 2023-2025")
print("Testing period      : January-September 2026")
print("Forecast origins    :", len(forecast_origins))
print("Forecast horizon    :", FORECAST_DAYS, "days")
print("Forecast steps      :", FORECAST_STEPS)
print("Samples per day     :", SAMPLES_PER_DAY)

print("\nOverall valid pairs :", test_metrics["N_VALID"])
print("Overall missing pairs:", test_metrics["N_MISSING"])
print("\nCell 8 - Final TEST results and metrics saved successfully.")

HGBR model M6.0 - FINAL TEST METRICS


,PERIOD,N_TOTAL,N_VALID,N_MISSING,MAE,RMSE,R,R2
0,Overall,78048,77347,701,4.6201,5.8573,0.9620,0.9036
1,D+1,26016,25782,234,4.0307,5.2246,0.9692,0.9234
2,D+2,26016,25782,234,4.6451,5.8441,0.9626,0.9040
3,D+3,26016,25783,233,5.1845,6.4399,0.9566,0.8833


HGBR model M6.0 - FINAL TEST RESULTS
Number of forecast origins: 271
Number of forecast steps : 288
DataFrame shape           : (78048, 6)


,FORECAST_ORIGIN,TARGET_TIME,FORECAST_DAY,FORECAST_STEP,OBSERVATION,PREDICTION
0,2026-01-01,2026-01-01 00:00:00,1,1,40.8200,40.5664
1,2026-01-01,2026-01-01 00:15:00,1,2,42.0700,43.2750
2,2026-01-01,2026-01-01 00:30:00,1,3,45.6800,44.9857
3,2026-01-01,2026-01-01 00:45:00,1,4,49.3400,47.3803
4,2026-01-01,2026-01-01 01:00:00,1,5,52.5900,49.4300
5,2026-01-01,2026-01-01 01:15:00,1,6,54.6100,51.3505
6,2026-01-01,2026-01-01 01:30:00,1,7,55.4500,53.3491
7,2026-01-01,2026-01-01 01:45:00,1,8,54.5300,55.8579
8,2026-01-01,2026-01-01 02:00:00,1,9,56.3700,57.6200
9,2026-01-01,2026-01-01 02:15:00,1,10,58.1900,59.9097



Timestamp verification:
First forecast origin : 2026-01-01 00:00:00
First target time     : 2026-01-01 00:00:00
Last target time      : 2026-01-03 23:45:00

Final TEST result files saved:
TXT: C:\ML\Data_Learn\HGBR_M6-0_Daily_VTEC_Forecast_Output\FINAL_TEST_Results.txt
CSV: C:\ML\Data_Learn\HGBR_M6-0_Daily_VTEC_Forecast_Output\FINAL_TEST_Results.csv

Final TEST metrics saved:
TXT: C:\ML\Data_Learn\HGBR_M6-0_Daily_VTEC_Forecast_Output\FINAL_TEST_Metrics.txt
CSV: C:\ML\Data_Learn\HGBR_M6-0_Daily_VTEC_Forecast_Output\FINAL_TEST_Metrics.csv
HGBR model M6.0 - FINAL EXPORT SUMMARY
Training mode       : final
Training period     : 2023-2025
Testing period      : January-September 2026
Forecast origins    : 271
Forecast horizon    : 3 days
Forecast steps      : 288
Samples per day     : 96

Overall valid pairs : 77347
Overall missing pairs: 701

Cell 8 - Final TEST results and metrics saved successfully.


In [9]:
# ============================================================
# CELL 9 — DATASET SAMPLE SUMMARY
# ============================================================

import numpy as np
def summarize_dataset(name, X, y):
    X = np.asarray(X)
    y = np.asarray(y)

    n_samples = len(X)
    # Sampel dengan seluruh input features tanpa NaN
    complete_input = np.isfinite(X).all(axis=1).sum()
    # Sampel dengan seluruh target forecast tanpa NaN
    complete_target = np.isfinite(y).all(axis=1).sum()
    # Jumlah target valid untuk setiap forecast time step
    valid_per_model = np.isfinite(y).sum(axis=0)
    print(f"\n{name}")
    print(f"Total forecasting samples       : {n_samples}")
    print(f"Complete historical inputs     : {complete_input}")
    print(f"Incomplete historical inputs   : {n_samples - complete_input}")
    print(f"Complete 3-day target samples  : {complete_target}")
    print(f"Valid target samples (min)     : {valid_per_model.min()}")
    print(f"Valid target samples (max)     : {valid_per_model.max()}")
    print(f"Valid target samples (mean)    : {valid_per_model.mean():.2f}")


print("HGBR model M6.0 — FORECASTING DATASET SUMMARY")

summarize_dataset("TRAINING DATASET (2023–2024)", X_train, y_train)
summarize_dataset("VALIDATION DATASET (2025)", X_val, y_val)
summarize_dataset("TESTING DATASET (Jan–Sept 2026)", X_test, y_test)

print("TOTAL DATASET SUMMARY")

print(f"Total samples : {len(X_train) + len(X_val) + len(X_test)}")
print(f"Training      : {len(X_train)}")
print(f"Validation    : {len(X_val)}")
print(f"Testing       : {len(X_test)}")


HGBR model M6.0 — FORECASTING DATASET SUMMARY

TRAINING DATASET (2023–2024)
Total forecasting samples       : 725
Complete historical inputs     : 74
Incomplete historical inputs   : 651
Complete 3-day target samples  : 491
Valid target samples (min)     : 656
Valid target samples (max)     : 684
Valid target samples (mean)    : 667.68

VALIDATION DATASET (2025)
Total forecasting samples       : 363
Complete historical inputs     : 74
Incomplete historical inputs   : 289
Complete 3-day target samples  : 265
Valid target samples (min)     : 350
Valid target samples (max)     : 362
Valid target samples (mean)    : 355.96

TESTING DATASET (Jan–Sept 2026)
Total forecasting samples       : 271
Complete historical inputs     : 60
Incomplete historical inputs   : 211
Complete 3-day target samples  : 221
Valid target samples (min)     : 263
Valid target samples (max)     : 271
Valid target samples (mean)    : 268.57
TOTAL DATASET SUMMARY
Total samples : 1359
Training      : 725
Validation    :

In [14]:
import sys
import sklearn

print("Python version:", sys.version)
print("scikit-learn version:", sklearn.__version__)

Python version: 3.9.11 (tags/v3.9.11:2de452f, Mar 16 2022, 14:33:45) [MSC v.1929 64 bit (AMD64)]
scikit-learn version: 1.0.2
